In [ ]:

# 41단계: Google Drive 연결 및 데이터 파일 확인

from google.colab import drive
from pathlib import Path
import pyarrow.parquet as pq
import json

# Google Drive 연결
drive.mount("/content/drive")

# 이전 코랩에서 저장한 데이터 폴더
DATA_DIR = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data"
)

# 파일 경로
NORMAL_PATH = DATA_DIR / "rf_normal_candidates.parquet"
TAMPERED_PATH = DATA_DIR / "rf_tampered_item_raw.parquet"
MANIFEST_PATH = DATA_DIR / "rf_handoff_manifest.json"
SPLIT_PATH = DATA_DIR / "rf_split_users.csv"

print("===== 41단계: 데이터 파일 확인 =====")

files = {
    "정상 후보": NORMAL_PATH,
    "합성 변조": TAMPERED_PATH,
    "이전 정보": MANIFEST_PATH,
    "사용자 분리": SPLIT_PATH
}

for name, path in files.items():
    print(f"{name}: {path.exists()}")

print("\n===== Parquet 메타데이터 =====")

for name, path in [
    ("정상 후보", NORMAL_PATH),
    ("합성 변조", TAMPERED_PATH)
]:
    if path.exists():
        metadata = pq.ParquetFile(str(path)).metadata
        print(
            f"{name}: {metadata.num_rows:,}건, "
            f"{metadata.num_columns}개 컬럼"
        )

if MANIFEST_PATH.exists():
    print("\n===== 이전 작업 정보 =====")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    print(json.dumps(manifest, ensure_ascii=False, indent=2))


Mounted at /content/drive
===== 41단계: 데이터 파일 확인 =====
정상 후보: True
합성 변조: True
이전 정보: True
사용자 분리: False

===== Parquet 메타데이터 =====
정상 후보: 99,998건, 28개 컬럼
합성 변조: 1,000건, 29개 컬럼

===== 이전 작업 정보 =====
{
  "normal_rows": 99998,
  "tampered_rows": 1000,
  "normal_columns": 28,
  "tampered_columns": 29,
  "synthetic_tamper_type": "item_and_rank_change",
  "pity_features_need_recalculation": true,
  "split_users_saved": false
}


In [ ]:

# 42단계: 랜덤 포레스트용 데이터 불러오기

import pandas as pd

# 정상 후보 데이터
normal_df = pd.read_parquet(NORMAL_PATH)

# 합성 변조 데이터
tampered_df = pd.read_parquet(TAMPERED_PATH)

print("===== 데이터 불러오기 결과 =====")
print("정상 후보 기록:", len(normal_df))
print("합성 변조 기록:", len(tampered_df))

print("\n===== 컬럼 확인 =====")
print("정상 후보 컬럼:", len(normal_df.columns))
print("합성 변조 컬럼:", len(tampered_df.columns))

print("\n===== 라벨 분포 =====")
print("정상 후보:")
print(normal_df["label"].value_counts())

print("\n합성 변조:")
print(tampered_df["label"].value_counts())

print("\n===== 사용자 수 =====")
print("정상 후보 사용자:", normal_df["user_id"].nunique())
print("합성 변조 사용자:", tampered_df["user_id"].nunique())

print("\n===== 데이터 미리보기 =====")
display(
    tampered_df[
        ["user_id", "name_ko", "rank_type",
         "gacha_type_ko", "label"]
    ].head(10)
)


===== 데이터 불러오기 결과 =====
정상 후보 기록: 99998
합성 변조 기록: 1000

===== 컬럼 확인 =====
정상 후보 컬럼: 28
합성 변조 컬럼: 29

===== 라벨 분포 =====
정상 후보:
label
0    99998
Name: count, dtype: int64

합성 변조:
label
1    1000
Name: count, dtype: int64

===== 사용자 수 =====
정상 후보 사용자: 116
합성 변조 사용자: 110

===== 데이터 미리보기 =====


,user_id,name_ko,rank_type,gacha_type_ko,label
0,db0efa3460469199,카구라의 진의,5,무기 이벤트 기원,1
1,c6fc3bbe418e2994,천공의 날개,5,캐릭터 이벤트 기원 2,1
2,e1342a5cebc31457,관홍의 창,5,무기 이벤트 기원,1
3,f83074c372574058,쇄석의 붉은 뿔,5,무기 이벤트 기원,1
4,3192234dbcd11252,쇄석의 붉은 뿔,5,캐릭터 이벤트 기원 1,1
5,c6fc3bbe418e2994,천공의 마루,5,무기 이벤트 기원,1
6,0a6fdc46ba544722,카구라의 진의,5,캐릭터 이벤트 기원 1,1
7,94d8cf4ddf394012,송뢰가 울릴 무렵,5,무기 이벤트 기원,1
8,e1342a5cebc31457,안개를 가르는 회광,5,무기 이벤트 기원,1
9,22484acc00d20881,카구라의 진의,5,무기 이벤트 기원,1


In [ ]:

# 43단계: 사용자 분리 정보 복구 및 검증

import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# 전체 원본 데이터 경로
SOURCE_PATH = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/genshin_processed/"
    "genshin_analysis_ready.parquet"
)

print("원본 파일 존재:", SOURCE_PATH.exists())

if SOURCE_PATH.exists():

    pf = pq.ParquetFile(str(SOURCE_PATH))

    # user_id 컬럼만 행 그룹별로 읽기
    all_users = set()

    for i in range(pf.num_row_groups):
        table = pf.read_row_group(
            i, columns=["user_id"]
        )
        all_users.update(table.column("user_id").to_pylist())

    user_list = sorted(all_users)

    # 기존과 동일한 분할 방식
    train_users, temp_users = train_test_split(
        user_list,
        test_size=0.30,
        random_state=42
    )

    val_users, test_users = train_test_split(
        temp_users,
        test_size=0.50,
        random_state=42
    )

    train_users = set(train_users)
    val_users = set(val_users)
    test_users = set(test_users)

    print("\n===== 사용자 분리 복구 결과 =====")
    print("전체 사용자:", len(all_users))
    print("학습 사용자:", len(train_users))
    print("검증 사용자:", len(val_users))
    print("테스트 사용자:", len(test_users))

    print("\n===== 사용자 중복 검사 =====")
    print("학습-검증:", len(train_users & val_users))
    print("학습-테스트:", len(train_users & test_users))
    print("검증-테스트:", len(val_users & test_users))

    print("\n===== 기존 학습 데이터 일치 검사 =====")
    normal_ids = set(normal_df["user_id"])
    tampered_ids = set(tampered_df["user_id"])

    print("정상 후보의 학습 그룹 외 사용자:",
          len(normal_ids - train_users))

    print("합성 변조의 학습 그룹 외 사용자:",
          len(tampered_ids - train_users))

else:
    print("원본 Parquet 파일을 찾지 못했습니다.")


원본 파일 존재: True

===== 사용자 분리 복구 결과 =====
전체 사용자: 16289
학습 사용자: 11402
검증 사용자: 2443
테스트 사용자: 2444

===== 사용자 중복 검사 =====
학습-검증: 0
학습-테스트: 0
검증-테스트: 0

===== 기존 학습 데이터 일치 검사 =====
정상 후보의 학습 그룹 외 사용자: 0
합성 변조의 학습 그룹 외 사용자: 0


In [ ]:

# 44단계: 사용자 분리 정보 저장 및 검증

import pandas as pd

# 사용자 분리 목록을 하나의 데이터프레임으로 정리
split_df = pd.concat([
    pd.DataFrame({
        "user_id": sorted(train_users),
        "split": "train"
    }),
    pd.DataFrame({
        "user_id": sorted(val_users),
        "split": "val"
    }),
    pd.DataFrame({
        "user_id": sorted(test_users),
        "split": "test"
    })
], ignore_index=True)

# 기존에 41단계에서 지정한 경로 사용
split_df.to_csv(
    SPLIT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# 저장된 파일 다시 읽어 검증
saved_split_df = pd.read_csv(
    SPLIT_PATH,
    encoding="utf-8-sig",
    dtype={"user_id": str}
)

print("===== 사용자 분리 정보 저장 결과 =====")
print("파일 존재:", SPLIT_PATH.exists())
print("저장 경로:", SPLIT_PATH)
print("전체 사용자:", len(saved_split_df))

print("\n===== 그룹별 사용자 수 =====")
print(saved_split_df["split"].value_counts())

print("\n===== 중복 검사 =====")
print(
    "중복 사용자:",
    saved_split_df["user_id"].duplicated().sum()
)

print("\n===== 기존 분리 정보 일치 검사 =====")
print(
    "학습 그룹 일치:",
    set(saved_split_df.loc[
        saved_split_df["split"] == "train", "user_id"
    ]) == train_users
)

print(
    "검증 그룹 일치:",
    set(saved_split_df.loc[
        saved_split_df["split"] == "val", "user_id"
    ]) == val_users
)

print(
    "테스트 그룹 일치:",
    set(saved_split_df.loc[
        saved_split_df["split"] == "test", "user_id"
    ]) == test_users
)


===== 사용자 분리 정보 저장 결과 =====
파일 존재: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/rf_split_users.csv
전체 사용자: 16289

===== 그룹별 사용자 수 =====
split
train    11402
test      2444
val       2443
Name: count, dtype: int64

===== 중복 검사 =====
중복 사용자: 0

===== 기존 분리 정보 일치 검사 =====
학습 그룹 일치: True
검증 그룹 일치: True
테스트 그룹 일치: True


In [ ]:

# 45단계: 합성 변조 기록과 원본 기록 연결 검사

import pandas as pd

# 원본 기록의 식별자와 주요 정보
original_records = normal_df[
    ["user_id", "gacha_id", "name_ko",
     "rank_type", "gacha_time", "gacha_type"]
].copy()

# 합성 변조 기록의 식별자와 주요 정보
tampered_records = tampered_df[
    ["user_id", "gacha_id", "name_ko",
     "rank_type", "gacha_time", "gacha_type"]
].copy()

# 사용자 ID + 기원 기록 ID로 연결
matched_df = tampered_records.merge(
    original_records,
    on=["user_id", "gacha_id"],
    how="left",
    suffixes=("_tampered", "_original"),
    indicator=True,
    validate="one_to_one"
)

print("===== 원본 연결 검사 =====")
print("전체 합성 변조:", len(tampered_df))

print("원본 연결 성공:",
      (matched_df["_merge"] == "both").sum())

print("원본 연결 실패:",
      (matched_df["_merge"] != "both").sum())

print("\n===== 변조 내용 검사 =====")
print("아이템명 변경:",
      (matched_df["name_ko_tampered"]
       != matched_df["name_ko_original"]).sum())

print("등급 변경:",
      (matched_df["rank_type_tampered"]
       != matched_df["rank_type_original"]).sum())

print("획득 시간 변경:",
      (matched_df["gacha_time_tampered"]
       != matched_df["gacha_time_original"]).sum())

print("배너 변경:",
      (matched_df["gacha_type_tampered"]
       != matched_df["gacha_type_original"]).sum())

print("\n===== 변조 전후 예시 =====")
display(
    matched_df[
        ["name_ko_original", "name_ko_tampered",
         "rank_type_original", "rank_type_tampered"]
    ].head(10)
)


===== 원본 연결 검사 =====
전체 합성 변조: 1000
원본 연결 성공: 1000
원본 연결 실패: 0

===== 변조 내용 검사 =====
아이템명 변경: 1000
등급 변경: 1000
획득 시간 변경: 0
배너 변경: 0

===== 변조 전후 예시 =====


,name_ko_original,name_ko_tampered,rank_type_original,rank_type_tampered
0,강철의 그림자,카구라의 진의,3,5
1,강철의 그림자,천공의 날개,3,5
2,드래곤 블러드 소드,관홍의 창,3,5
3,신궁의 서약,쇄석의 붉은 뿔,3,5
4,여명신검,쇄석의 붉은 뿔,3,5
5,신궁의 서약,천공의 마루,3,5
6,드래곤 블러드 소드,카구라의 진의,3,5
7,신궁의 서약,송뢰가 울릴 무렵,3,5
8,드래곤 슬레이어 영웅담,안개를 가르는 회광,3,5
9,마도 서론,카구라의 진의,3,5


In [ ]:

# 46단계 수정본: 자료형 경고 없이 합성 변조 이력 생성

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

# 1. 정상 후보 데이터 복사
tampered_history_df = normal_df.copy()

# 2. 기록 식별자 중복 확인
assert not normal_df.duplicated(KEY_COLS).any(), \
    "정상 후보에 중복 식별자가 있습니다."

assert not tampered_df.duplicated(KEY_COLS).any(), \
    "변조 데이터에 중복 식별자가 있습니다."

# 3. 식별자를 인덱스로 설정
tampered_history_df = tampered_history_df.set_index(
    KEY_COLS, drop=False
)

replacement_df = tampered_df.set_index(
    KEY_COLS, drop=False
)

# 4. 모든 변조 기록이 원본에 존재하는지 확인
assert replacement_df.index.isin(
    tampered_history_df.index
).all(), "원본과 연결되지 않는 변조 기록이 있습니다."

# 5. 자료형에 맞게 컬럼별 교체
replace_cols = [
    "name_zh",
    "name_ko",
    "rank_type",
    "item_type",
    "item_type_ko",
    "is_five_star",
    "is_four_star"
]

for col in replace_cols:
    target_dtype = tampered_history_df[col].dtype

    new_values = replacement_df[col].astype(
        target_dtype
    )

    # 자료형을 유지하면서 해당 기록만 교체
    tampered_history_df.loc[
        replacement_df.index, col
    ] = new_values.to_numpy()

# 6. 인덱스 초기화
tampered_history_df = (
    tampered_history_df.reset_index(drop=True)
)

# 7. 데이터 검증
changed_count = (
    normal_df["rank_type"].to_numpy()
    != tampered_history_df["rank_type"].to_numpy()
).sum()

five_error = (
    tampered_history_df["is_five_star"]
    != (tampered_history_df["rank_type"] == 5)
).sum()

four_error = (
    tampered_history_df["is_four_star"]
    != (tampered_history_df["rank_type"] == 4)
).sum()

print("===== 수정된 46단계 실행 결과 =====")
print("전체 기록:", len(tampered_history_df))
print("사용자 수:", tampered_history_df["user_id"].nunique())
print("등급 변경 기록:", changed_count)

print("\n===== 등급별 기록 =====")
print(
    tampered_history_df["rank_type"]
    .value_counts()
    .sort_index()
)

print("\n===== 자료형 확인 =====")
print("rank_type:", tampered_history_df["rank_type"].dtype)
print("item_type:", tampered_history_df["item_type"].dtype)

print("\n===== 일관성 검사 =====")
print("5성 표시 불일치:", five_error)
print("4성 표시 불일치:", four_error)
print("원본 정상 후보:", len(normal_df))


===== 수정된 46단계 실행 결과 =====
전체 기록: 99998
사용자 수: 116
등급 변경 기록: 1000

===== 등급별 기록 =====
rank_type
3    84036
4    13313
5     2649
Name: count, dtype: int64

===== 자료형 확인 =====
rank_type: int8
item_type: int8

===== 일관성 검사 =====
5성 표시 불일치: 0
4성 표시 불일치: 0
원본 정상 후보: 99998


In [ ]:

# 47단계: 합성 변조 이력의 천장 변수 재계산

import pandas as pd
import numpy as np

# 기존 변조 이력 보존
recalculated_df = tampered_history_df.copy()

group_cols = ["user_id", "banner_group"]

# 사용자별 + 천장 공유 배너별 정렬
recalculated_df = recalculated_df.sort_values(
    ["user_id", "banner_group",
     "gacha_time", "gacha_id_num"],
    kind="stable"
).reset_index(drop=True)

# 그룹 내 뽑기 순서
recalculated_df["group_order"] = (
    recalculated_df.groupby(group_cols).cumcount()
)

# 5성 기록 위치
five_positions = recalculated_df["group_order"].where(
    recalculated_df["rank_type"] == 5
)

# 가장 최근 5성 위치
last_five = five_positions.groupby(
    [
        recalculated_df["user_id"],
        recalculated_df["banner_group"]
    ]
).ffill()

# 마지막 5성 이후 누적 횟수
recalculated_df["pulls_since_last_5star"] = (
    recalculated_df["group_order"] - last_five
)

# 현재 뽑기 이전의 천장 상태
recalculated_df["pity_before_pull"] = (
    recalculated_df.groupby(group_cols)[
        "pulls_since_last_5star"
    ].shift(1)
)

# 현재 뽑기를 포함한 횟수
recalculated_df["current_pity_count"] = (
    recalculated_df["pity_before_pull"] + 1
)

print("===== 천장 변수 재계산 결과 =====")
print("전체 기록:", len(recalculated_df))
print("5성 기록:", (recalculated_df["rank_type"] == 5).sum())

print("\n===== 천장 변수 통계 =====")
print(
    recalculated_df[
        "pulls_since_last_5star"
    ].describe()
)

print("\n===== 직전 천장 상태 결측값 =====")
print(
    recalculated_df[
        "pity_before_pull"
    ].isna().sum()
)

print("\n===== 5성 기록 초기화 검사 =====")
five_error = (
    (recalculated_df["rank_type"] == 5)
    & recalculated_df["pulls_since_last_5star"].notna()
    & (recalculated_df["pulls_since_last_5star"] != 0)
)

print("5성 획득 후 누적 횟수 오류:", five_error.sum())


===== 천장 변수 재계산 결과 =====
전체 기록: 99998
5성 기록: 2649

===== 천장 변수 통계 =====
count    88035.000000
mean        25.948702
std         20.196526
min          0.000000
25%          9.000000
50%         21.000000
75%         40.000000
max         83.000000
Name: pulls_since_last_5star, dtype: float64

===== 직전 천장 상태 결측값 =====
12287

===== 5성 기록 초기화 검사 =====
5성 획득 후 누적 횟수 오류: 0


In [ ]:

# 48단계: 재계산된 천장 규칙 검사

import pandas as pd

# 배너 그룹별 천장 기준
pity_limits = {
    "character_event": 90,
    "weapon_event": 80,
    "standard": 90
}

# 새로운 천장 기준 연결
recalculated_df["pity_limit_new"] = (
    recalculated_df["banner_group"].map(pity_limits)
)

# 재계산된 천장 초과 여부
recalculated_df["pity_violation_new"] = (
    recalculated_df["current_pity_count"]
    > recalculated_df["pity_limit_new"]
).fillna(False)

print("===== 재계산된 천장 규칙 검사 =====")
print("전체 기록:", len(recalculated_df))

print(
    "천장 초과 의심 기록:",
    int(recalculated_df["pity_violation_new"].sum())
)

print("\n===== 배너별 천장 초과 =====")
display(
    recalculated_df.groupby("banner_group")[
        "pity_violation_new"
    ].agg(["size", "sum"])
)

print("\n===== 천장 초과 기록 예시 =====")
display(
    recalculated_df.loc[
        recalculated_df["pity_violation_new"],
        [
            "user_id",
            "banner_group",
            "gacha_time",
            "rank_type",
            "current_pity_count",
            "pity_limit_new"
        ]
    ].head(15)
)

print("\n===== 5성 기록 확인 =====")
print(
    "재계산 후 5성 기록:",
    (recalculated_df["rank_type"] == 5).sum()
)


===== 재계산된 천장 규칙 검사 =====
전체 기록: 99998
천장 초과 의심 기록: 0

===== 배너별 천장 초과 =====


,size,sum
banner_group,,
beginner,960,0
character_event,66687,0
standard,15723,0
weapon_event,16628,0



===== 천장 초과 기록 예시 =====


,user_id,banner_group,gacha_time,rank_type,current_pity_count,pity_limit_new



===== 5성 기록 확인 =====
재계산 후 5성 기록: 2649


In [ ]:

# 49단계: 랜덤 포레스트 입력 변수 점검

import pandas as pd

# 학습에 사용할 후보 변수
candidate_features = [
    "gacha_type",
    "rank_type",
    "item_type",
    "time_diff_seconds",
    "pity_before_pull",
    "pull_order",
    "banner_pull_order"
]

print("===== 모델 입력 후보 변수 =====")

feature_info = []

for col in candidate_features:

    if col in recalculated_df.columns:
        feature_info.append({
            "변수": col,
            "자료형": str(recalculated_df[col].dtype),
            "결측값": int(recalculated_df[col].isna().sum()),
            "고유값 수": int(recalculated_df[col].nunique())
        })

    else:
        print("존재하지 않는 변수:", col)

feature_info_df = pd.DataFrame(feature_info)

display(feature_info_df)

print("\n===== 모델 입력에서 제외할 변수 =====")

excluded_features = [
    "user_id",
    "gacha_id",
    "label",
    "tamper_type",
    "pulls_since_last_5star",
    "pity_violation_new"
]

for col in excluded_features:
    print("-", col)

print("\n===== 데이터 검증 =====")
print("전체 기록:", len(recalculated_df))
print("사용자 수:", recalculated_df["user_id"].nunique())
print("5성 기록:", (recalculated_df["rank_type"] == 5).sum())


===== 모델 입력 후보 변수 =====


,변수,자료형,결측값,고유값 수
0,gacha_type,object,0,5
1,rank_type,int8,0,3
2,item_type,int8,0,2
3,time_diff_seconds,float64,383,17183
4,pity_before_pull,float64,12287,84
5,pull_order,int64,0,3856
6,banner_pull_order,int64,0,2134



===== 모델 입력에서 제외할 변수 =====
- user_id
- gacha_id
- label
- tamper_type
- pulls_since_last_5star
- pity_violation_new

===== 데이터 검증 =====
전체 기록: 99998
사용자 수: 116
5성 기록: 2649


In [ ]:

# 50단계: 합성 변조 기록에 정답 라벨 적용

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

# 재계산 데이터 보존
labeled_df = recalculated_df.copy()

# 변조 기록 식별자
tampered_keys = pd.MultiIndex.from_frame(
    tampered_df[KEY_COLS]
)

# 전체 기록 식별자
all_keys = pd.MultiIndex.from_frame(
    labeled_df[KEY_COLS]
)

# 변조 여부 확인
is_tampered = all_keys.isin(tampered_keys)

# 라벨 부여
labeled_df["label"] = is_tampered.astype("int8")

print("===== 라벨 적용 결과 =====")
print("전체 기록:", len(labeled_df))

print("\n===== 라벨별 기록 수 =====")
print(labeled_df["label"].value_counts().sort_index())

print("\n===== 검증 =====")
print("변조 라벨 수:", int(labeled_df["label"].sum()))
print("원본 식별자 중복:",
      labeled_df.duplicated(KEY_COLS).sum())

print("\n===== 변조 기록 예시 =====")
display(
    labeled_df.loc[
        labeled_df["label"] == 1,
        ["user_id", "name_ko",
         "rank_type", "banner_group", "label"]
    ].head(10)
)


===== 라벨 적용 결과 =====
전체 기록: 99998

===== 라벨별 기록 수 =====
label
0    98998
1     1000
Name: count, dtype: int64

===== 검증 =====
변조 라벨 수: 1000
원본 식별자 중복: 0

===== 변조 기록 예시 =====


,user_id,name_ko,rank_type,banner_group,label
309,010e0c3a2cb39684,천공의 긍지,5,character_event,1
374,010e0c3a2cb39684,송뢰가 울릴 무렵,5,character_event,1
396,010e0c3a2cb39684,하란 월백의 후츠,5,character_event,1
652,03cebbd7ca155748,오래된 자유의 서약,5,character_event,1
673,03cebbd7ca155748,식재,5,character_event,1
716,03cebbd7ca155748,무공의 검,5,character_event,1
728,03cebbd7ca155748,오래된 자유의 서약,5,character_event,1
826,03cebbd7ca155748,반암결록,5,character_event,1
1112,03cebbd7ca155748,카구라의 진의,5,character_event,1
1165,03cebbd7ca155748,매의 검,5,character_event,1


In [ ]:

# 51단계: 등급별 정상·변조 라벨 분포 분석

import pandas as pd

# 등급별 정상/변조 기록 수
rank_label_table = pd.crosstab(
    labeled_df["rank_type"],
    labeled_df["label"]
)

rank_label_table = rank_label_table.rename(
    columns={0: "정상 후보", 1: "합성 변조"}
)

print("===== 등급별 라벨 분포 =====")
display(rank_label_table)

# 등급별 변조 비율
rank_tamper_ratio = (
    labeled_df.groupby("rank_type")["label"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "전체 기록",
        "sum": "변조 기록",
        "mean": "변조 비율"
    })
)

rank_tamper_ratio["변조 비율"] = (
    rank_tamper_ratio["변조 비율"] * 100
).round(2)

print("\n===== 등급별 변조 비율 =====")
display(rank_tamper_ratio)

print("\n===== 전체 라벨 비율 =====")
print(
    (labeled_df["label"].value_counts(normalize=True) * 100)
    .round(2)
)


===== 등급별 라벨 분포 =====


label,정상 후보,합성 변조
rank_type,,
3,84036,0
4,13313,0
5,1649,1000



===== 등급별 변조 비율 =====


,전체 기록,변조 기록,변조 비율
rank_type,,,
3,84036,0,0.00
4,13313,0,0.00
5,2649,1000,37.75



===== 전체 라벨 비율 =====
label
0    99.0
1     1.0
Name: proportion, dtype: float64


In [ ]:

# 52단계: 합성 변조 다양화를 위한 정상 후보 분포 확인

import pandas as pd

# 변조되지 않은 기록만 추출
normal_candidates_df = labeled_df.loc[
    labeled_df["label"] == 0
].copy()

print("===== 정상 후보 데이터 =====")
print("전체 기록:", len(normal_candidates_df))
print("사용자 수:", normal_candidates_df["user_id"].nunique())

print("\n===== 등급 및 아이템 종류별 기록 =====")
rank_item_table = pd.crosstab(
    normal_candidates_df["rank_type"],
    normal_candidates_df["item_type_ko"]
)

display(rank_item_table)

print("\n===== 배너별 정상 후보 기록 =====")
display(
    normal_candidates_df[
        "gacha_type_ko"
    ].value_counts()
)

print("\n===== 4성 기록 예시 =====")
display(
    normal_candidates_df.loc[
        normal_candidates_df["rank_type"] == 4,
        ["name_ko", "item_type_ko",
         "gacha_type_ko", "rank_type"]
    ].head(10)
)


===== 정상 후보 데이터 =====
전체 기록: 98998
사용자 수: 116

===== 등급 및 아이템 종류별 기록 =====


item_type_ko,무기,캐릭터
rank_type,,
3,84036,0
4,5516,7797
5,447,1202



===== 배너별 정상 후보 기록 =====


,count
gacha_type_ko,
캐릭터 이벤트 기원 1,51849
무기 이벤트 기원,16470
상시 기원,15568
캐릭터 이벤트 기원 2,14160
초보자 기원,951



===== 4성 기록 예시 =====


,name_ko,item_type_ko,gacha_type_ko,rank_type
7,노엘,캐릭터,초보자 기원,4
17,행추,캐릭터,초보자 기원,4
25,향릉,캐릭터,캐릭터 이벤트 기원 1,4
35,노엘,캐릭터,캐릭터 이벤트 기원 1,4
41,북두,캐릭터,캐릭터 이벤트 기원 2,4
47,행추,캐릭터,캐릭터 이벤트 기원 2,4
56,제례 대검,무기,캐릭터 이벤트 기원 2,4
65,연비,캐릭터,캐릭터 이벤트 기원 2,4
75,페보니우스 장창,무기,캐릭터 이벤트 기원 2,4
84,행추,캐릭터,캐릭터 이벤트 기원 2,4


In [ ]:

# 53단계: 4성 캐릭터 변조를 위한 후보 목록 확인

import pandas as pd

# 정상 후보 중 4성 캐릭터만 추출
four_star_characters = normal_candidates_df.loc[
    (normal_candidates_df["rank_type"] == 4)
    & (normal_candidates_df["item_type_ko"] == "캐릭터")
].copy()

# 캐릭터별 기록 수 확인
character_stats = (
    four_star_characters
    .groupby("name_ko")
    .agg(
        record_count=("name_ko", "size"),
        first_date=("gacha_time", "min"),
        last_date=("gacha_time", "max")
    )
    .reset_index()
    .sort_values("record_count", ascending=False)
)

print("===== 4성 캐릭터 분석 =====")
print("전체 기록:", len(four_star_characters))
print("캐릭터 종류:", len(character_stats))

print("\n===== 캐릭터별 기록 수 =====")
display(character_stats)

print("\n===== 배너별 기록 수 =====")
print(four_star_characters["gacha_type_ko"].value_counts())

print("\n===== 원본 보존 확인 =====")
print("정상 후보:", len(normal_candidates_df))
print("기존 합성 변조:", int(labeled_df["label"].sum()))


===== 4성 캐릭터 분석 =====
전체 기록: 7797
캐릭터 종류: 25

===== 캐릭터별 기록 수 =====


,name_ko,record_count,first_date,last_date
14,연비,579,2021-04-28 12:09:43,2022-07-23 23:17:50
24,향릉,480,2021-01-22 20:31:16,2022-07-13 15:16:55
7,베넷,460,2021-06-23 01:00:32,2022-07-13 10:54:34
23,행추,459,2021-01-18 22:49:24,2022-07-21 21:20:26
8,북두,442,2021-02-03 11:08:36,2022-07-13 11:03:53
19,쿠죠 사라,433,2021-09-01 10:09:03,2022-07-13 14:19:03
2,디오나,429,2021-02-09 23:03:22,2022-07-13 11:13:22
21,토마,408,2021-11-02 18:00:36,2022-07-23 23:51:18
4,로자리아,394,2021-04-12 07:44:54,2022-07-14 19:44:49
10,설탕,389,2021-06-09 12:39:57,2022-07-18 18:44:40



===== 배너별 기록 수 =====
gacha_type_ko
캐릭터 이벤트 기원 1    4859
캐릭터 이벤트 기원 2    1346
상시 기원           1021
무기 이벤트 기원        441
초보자 기원           130
Name: count, dtype: int64

===== 원본 보존 확인 =====
정상 후보: 98998
기존 합성 변조: 1000


In [ ]:

# 54단계: 4성 캐릭터 → 다른 4성 캐릭터 변조

import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

# 원본 정상 후보에서 4성 캐릭터 500건 추출
target_char_df = normal_df.loc[
    (normal_df["rank_type"] == 4) &
    (normal_df["item_type_ko"] == "캐릭터")
].sample(n=500, random_state=42).copy()

# 캐릭터 후보 사전 (중국어/한국어 이름 쌍 유지)
char_pool = (
    normal_df.loc[
        (normal_df["rank_type"] == 4) &
        (normal_df["item_type_ko"] == "캐릭터"),
        ["name_zh", "name_ko"]
    ]
    .drop_duplicates(subset=["name_ko"])
    .reset_index(drop=True)
)

# 원본과 다른 캐릭터를 선택
new_names_zh = []
new_names_ko = []

for original_name in target_char_df["name_ko"]:
    candidates = char_pool.loc[
        char_pool["name_ko"] != original_name
    ]

    selected = candidates.iloc[
        rng.integers(len(candidates))
    ]

    new_names_zh.append(selected["name_zh"])
    new_names_ko.append(selected["name_ko"])

# 별도의 변조 복사본 생성
tampered_char_df = target_char_df.copy()

tampered_char_df["name_zh"] = new_names_zh
tampered_char_df["name_ko"] = new_names_ko
tampered_char_df["label"] = 1
tampered_char_df["tamper_type"] = "character_name_change"

print("===== 4성 캐릭터 변조 결과 =====")
print("변조 기록:", len(tampered_char_df))

print("등급 유지:",
      (tampered_char_df["rank_type"] == 4).sum())

print("아이템명 변경:",
      (target_char_df["name_ko"].to_numpy()
       != tampered_char_df["name_ko"].to_numpy()).sum())

print("\n===== 변조 전후 예시 =====")
display(pd.DataFrame({
    "원본 캐릭터": target_char_df["name_ko"].to_numpy(),
    "변조 캐릭터": tampered_char_df["name_ko"].to_numpy(),
    "등급": tampered_char_df["rank_type"].to_numpy()
}).head(15))

print("\n===== 원본 보존 =====")
print("정상 후보:", len(normal_df))
print("기존 합성 변조:", len(tampered_df))


===== 4성 캐릭터 변조 결과 =====
변조 기록: 500
등급 유지: 500
아이템명 변경: 500

===== 변조 전후 예시 =====


,원본 캐릭터,변조 캐릭터,등급
0,피슬,향릉,4
1,연비,피슬,4
2,설탕,쿠죠 사라,4
3,운근,사유,4
4,바바라,로자리아,4
5,설탕,엠버,4
6,행추,북두,4
7,토마,설탕,4
8,연비,토마,4
9,디오나,향릉,4



===== 원본 보존 =====
정상 후보: 99998
기존 합성 변조: 1000


In [ ]:

# 55단계: 4성 캐릭터 변경 후 등장 시기 검사

import pandas as pd

# 1. 원본 데이터에서 캐릭터별 최초 관측 날짜 계산
first_seen = (
    normal_df.loc[
        (normal_df["rank_type"] == 4) &
        (normal_df["item_type_ko"] == "캐릭터")
    ]
    .groupby("name_ko")["gacha_time"]
    .min()
)

# 2. 변조된 캐릭터의 최초 관측 날짜 연결
tampered_char_df["first_seen_date"] = (
    tampered_char_df["name_ko"].map(first_seen)
)

# 3. 해당 캐릭터의 최초 관측 이전인지 확인
tampered_char_df["before_first_seen"] = (
    tampered_char_df["gacha_time"]
    < tampered_char_df["first_seen_date"]
)

# 4. 결과 출력
print("===== 등장 시기 검사 결과 =====")
print("전체 변조 기록:", len(tampered_char_df))

print(
    "최초 관측 이전 기록:",
    int(tampered_char_df["before_first_seen"].sum())
)

print(
    "그 외 기록:",
    int((~tampered_char_df["before_first_seen"]).sum())
)

print(
    "최초 관측일 결측값:",
    int(tampered_char_df["first_seen_date"].isna().sum())
)

print("\n===== 등장 시기 의심 기록 예시 =====")

display(
    tampered_char_df.loc[
        tampered_char_df["before_first_seen"],
        [
            "name_ko",
            "gacha_time",
            "first_seen_date",
            "rank_type",
            "banner_group"
        ]
    ].head(15)
)

print("\n===== 원본 보존 확인 =====")
print("원본 정상 후보:", len(normal_df))
print("4성 캐릭터 합성 변조:", len(tampered_char_df))


===== 등장 시기 검사 결과 =====
전체 변조 기록: 500
최초 관측 이전 기록: 53
그 외 기록: 447
최초 관측일 결측값: 0

===== 등장 시기 의심 기록 예시 =====


,name_ko,gacha_time,first_seen_date,rank_type,banner_group
47768,토마,2021-10-29 00:22:19,2021-11-02 18:00:36,4,character_event
81873,쿠키 시노부,2022-02-01 13:16:45,2022-06-21 18:00:14,4,character_event
89243,고로,2021-08-16 18:13:32,2021-12-14 18:00:40,4,character_event
7065,운근,2021-09-12 12:32:08,2022-01-05 10:53:07,4,weapon_event
1565,시카노인 헤이조,2022-01-20 12:01:09,2022-07-13 10:51:17,4,character_event
22154,쿠키 시노부,2022-06-20 21:09:00,2022-06-21 18:00:14,4,character_event
71885,시카노인 헤이조,2021-10-22 17:11:17,2022-07-13 10:51:17,4,character_event
89292,운근,2021-09-21 18:20:47,2022-01-05 10:53:07,4,character_event
48047,쿠키 시노부,2021-11-14 17:46:18,2022-06-21 18:00:14,4,character_event
63489,시카노인 헤이조,2022-01-17 21:02:48,2022-07-13 10:51:17,4,character_event



===== 원본 보존 확인 =====
원본 정상 후보: 99998
4성 캐릭터 합성 변조: 500


In [ ]:

# 56단계: 두 합성 변조 데이터의 중복 및 원본 확인

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

# 기존 3성→5성 변조 식별자
rank_tamper_keys = pd.MultiIndex.from_frame(
    tampered_df[KEY_COLS]
)

# 새로운 4성 캐릭터 변조 식별자
character_tamper_keys = pd.MultiIndex.from_frame(
    tampered_char_df[KEY_COLS]
)

# 변조 데이터 내부 중복 검사
rank_duplicates = tampered_df.duplicated(KEY_COLS).sum()
character_duplicates = tampered_char_df.duplicated(KEY_COLS).sum()

# 서로 겹치는 원본 기록 확인
overlap_count = character_tamper_keys.isin(
    rank_tamper_keys
).sum()

# 정상 원본에 존재하는 기록인지 확인
normal_keys = pd.MultiIndex.from_frame(
    normal_df[KEY_COLS]
)

rank_matched = rank_tamper_keys.isin(normal_keys).sum()
character_matched = character_tamper_keys.isin(normal_keys).sum()

print("===== 합성 변조 데이터 중복 검사 =====")
print("기존 등급+아이템 변조:", len(tampered_df))
print("신규 캐릭터 이름 변조:", len(tampered_char_df))

print("\n===== 중복 검사 =====")
print("기존 변조 내부 중복:", rank_duplicates)
print("신규 변조 내부 중복:", character_duplicates)
print("두 변조 유형 간 중복:", overlap_count)

print("\n===== 정상 원본 연결 검사 =====")
print("기존 변조 원본 연결:", rank_matched)
print("신규 변조 원본 연결:", character_matched)

print("\n===== 합성 변조 총계 =====")
print(
    "서로 다른 변조 대상 기록:",
    len(rank_tamper_keys.union(character_tamper_keys))
)


===== 합성 변조 데이터 중복 검사 =====
기존 등급+아이템 변조: 1000
신규 캐릭터 이름 변조: 500

===== 중복 검사 =====
기존 변조 내부 중복: 0
신규 변조 내부 중복: 0
두 변조 유형 간 중복: 0

===== 정상 원본 연결 검사 =====
기존 변조 원본 연결: 1000
신규 변조 원본 연결: 500

===== 합성 변조 총계 =====
서로 다른 변조 대상 기록: 1500


In [ ]:

# 57단계: 두 가지 합성 변조 유형 통합

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

# 1. 두 변조 유형 결합
rank_changes = tampered_df.copy()
rank_changes["tamper_type"] = "item_and_rank_change"

character_changes = tampered_char_df.copy()
character_changes["tamper_type"] = "character_name_change"

combined_tampered_df = pd.concat(
    [rank_changes, character_changes],
    ignore_index=True
)

# 2. 변조 대상의 중복 검사
assert not combined_tampered_df.duplicated(
    KEY_COLS
).any(), "변조 대상이 중복되었습니다."

# 3. 원본 이력 복사
combined_history_df = normal_df.copy()

# 4. 변조될 컬럼만 교체
replace_cols = [
    "name_zh",
    "name_ko",
    "rank_type",
    "item_type",
    "item_type_ko",
    "is_five_star",
    "is_four_star"
]

combined_history_df = combined_history_df.set_index(
    KEY_COLS, drop=False
)

replacement = combined_tampered_df.set_index(
    KEY_COLS, drop=False
)

assert replacement.index.isin(
    combined_history_df.index
).all(), "원본에 없는 변조 기록이 있습니다."

for col in replace_cols:
    target_dtype = combined_history_df[col].dtype
    values = replacement[col].astype(target_dtype)

    combined_history_df.loc[
        replacement.index, col
    ] = values.to_numpy()

# 5. 라벨과 변조 유형 적용
combined_history_df["label"] = 0
combined_history_df["tamper_type"] = "normal"

combined_history_df.loc[
    replacement.index, "label"
] = 1

combined_history_df.loc[
    replacement.index, "tamper_type"
] = replacement["tamper_type"].to_numpy()

combined_history_df = combined_history_df.reset_index(
    drop=True
)

# 6. 결과 검사
print("===== 57단계 통합 결과 =====")
print("전체 기록:", len(combined_history_df))
print("사용자 수:", combined_history_df["user_id"].nunique())

print("\n===== 라벨별 기록 수 =====")
print(combined_history_df["label"].value_counts().sort_index())

print("\n===== 변조 유형별 기록 수 =====")
print(combined_history_df["tamper_type"].value_counts())

print("\n===== 데이터 검증 =====")
print("중복 기록:",
      combined_history_df.duplicated(KEY_COLS).sum())

print("5성 표시 불일치:",
      (combined_history_df["is_five_star"]
       != (combined_history_df["rank_type"] == 5)).sum())

print("4성 표시 불일치:",
      (combined_history_df["is_four_star"]
       != (combined_history_df["rank_type"] == 4)).sum())


===== 57단계 통합 결과 =====
전체 기록: 99998
사용자 수: 116

===== 라벨별 기록 수 =====
label
0    98498
1     1500
Name: count, dtype: int64

===== 변조 유형별 기록 수 =====
tamper_type
normal                   98498
item_and_rank_change      1000
character_name_change      500
Name: count, dtype: int64

===== 데이터 검증 =====
중복 기록: 0
5성 표시 불일치: 0
4성 표시 불일치: 0


In [ ]:

# 57단계: 원신 공식 확률 기준표 생성 및 저장

import pandas as pd
from pathlib import Path

# 2021~2022년 기원 상세 정보 기준
official_rates_df = pd.DataFrame([
    {
        "banner_group": "character_event",
        "banner_name": "캐릭터 이벤트 기원",
        "base_rate": 0.006,
        "consolidated_rate": 0.016,
        "pity_limit": 90
    },
    {
        "banner_group": "weapon_event",
        "banner_name": "무기 이벤트 기원",
        "base_rate": 0.007,
        "consolidated_rate": 0.0185,
        "pity_limit": 80
    },
    {
        "banner_group": "standard",
        "banner_name": "상시 기원",
        "base_rate": 0.006,
        "consolidated_rate": 0.016,
        "pity_limit": 90
    }
])

# 저장 경로
save_dir = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data"
)
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "genshin_official_rates.csv"

# 파일 저장
official_rates_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

# 결과 출력
print("===== 57단계: 공식 확률 추가 =====")

display(
    official_rates_df.assign(
        base_rate=lambda x: x["base_rate"] * 100,
        consolidated_rate=lambda x: x["consolidated_rate"] * 100
    ).rename(columns={
        "base_rate": "기본 확률(%)",
        "consolidated_rate": "종합 확률(%)",
        "pity_limit": "천장 횟수"
    })
)

print("\n===== 파일 저장 확인 =====")
print("저장 경로:", save_path)
print("파일 존재:", save_path.exists())
print("등록된 기원 유형:", len(official_rates_df))

print("\n===== 기존 데이터 확인 =====")
print("원본 정상 후보:", len(normal_df))
print("기존 등급 변조:", len(tampered_df))
print("기존 캐릭터 변조:", len(tampered_char_df))


===== 57단계: 공식 확률 추가 =====


,banner_group,banner_name,기본 확률(%),종합 확률(%),천장 횟수
0,character_event,캐릭터 이벤트 기원,0.6,1.60,90
1,weapon_event,무기 이벤트 기원,0.7,1.85,80
2,standard,상시 기원,0.6,1.60,90



===== 파일 저장 확인 =====
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/genshin_official_rates.csv
파일 존재: True
등록된 기원 유형: 3

===== 기존 데이터 확인 =====
원본 정상 후보: 99998
기존 등급 변조: 1000
기존 캐릭터 변조: 500


In [ ]:

# 58단계: 원본 전체 데이터의 배너별 5성 획득 비율 계산

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# 1. 변조되지 않은 전체 원본 데이터
source_path = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2/"
    "genshin_processed/genshin_analysis_ready.parquet"
)

assert source_path.exists(), "원본 파일을 찾을 수 없습니다."

# 2. 기원 코드와 배너 유형 연결
banner_map = {
    "301": "character_event",
    "400": "character_event",
    "302": "weapon_event",
    "200": "standard",
    "100": "beginner"
}

# 3. 필요한 두 컬럼만 묶음 단위로 읽기
parquet_file = pq.ParquetFile(source_path)
result_parts = []
total_rows = 0

for i in range(parquet_file.num_row_groups):
    batch = parquet_file.read_row_group(
        i, columns=["gacha_type", "rank_type"]
    ).to_pandas()

    total_rows += len(batch)

    batch["banner_group"] = (
        batch["gacha_type"].astype(str).map(banner_map)
    )

    rank = pd.to_numeric(batch["rank_type"], errors="coerce")
    batch["is_5star"] = rank.eq(5).astype("int8")

    part = batch.groupby(
        "banner_group", dropna=False
    ).agg(
        total_pulls=("is_5star", "size"),
        five_star_count=("is_5star", "sum")
    ).reset_index()

    result_parts.append(part)

    if (i + 1) % 10 == 0 or i + 1 == parquet_file.num_row_groups:
        print(f"읽기 완료: {i + 1}/{parquet_file.num_row_groups}개 묶음")

# 4. 전체 집계
combined = pd.concat(result_parts, ignore_index=True)

actual_rates_df = (
    combined.groupby("banner_group", dropna=False)
    [["total_pulls", "five_star_count"]]
    .sum()
    .reset_index()
)

actual_rates_df["actual_rate"] = (
    actual_rates_df["five_star_count"]
    / actual_rates_df["total_pulls"]
)

# 5. 결과 저장
save_path = (
    Path("/content/drive/MyDrive/캡스톤 2/")
    / "capstonedata2/rf_model_data/genshin_actual_rates.csv"
)

actual_rates_df.to_csv(
    save_path, index=False, encoding="utf-8-sig"
)

# 6. 결과 확인
print("\n===== 58단계: 원본 전체 분석 결과 =====")
print("전체 원본 기록:", f"{total_rows:,}")
print("배너별 기록 합계:",
      f"{actual_rates_df['total_pulls'].sum():,}")

print("\n===== 배너별 실제 5성 획득 비율 =====")
display(
    actual_rates_df.assign(
        actual_rate=lambda x: (x["actual_rate"] * 100).round(4)
    ).rename(columns={
        "total_pulls": "뽑기 횟수",
        "five_star_count": "5성 획득 횟수",
        "actual_rate": "실제 획득 비율(%)"
    })
)

print("\n===== 저장 확인 =====")
print("파일 존재:", save_path.exists())
print("저장 경로:", save_path)


읽기 완료: 10/60개 묶음
읽기 완료: 20/60개 묶음
읽기 완료: 30/60개 묶음
읽기 완료: 40/60개 묶음
읽기 완료: 50/60개 묶음
읽기 완료: 60/60개 묶음

===== 58단계: 원본 전체 분석 결과 =====
전체 원본 기록: 15,598,804
배너별 기록 합계: 15,598,804

===== 배너별 실제 5성 획득 비율 =====


,banner_group,뽑기 횟수,5성 획득 횟수,실제 획득 비율(%)
0,beginner,112120,781,0.6966
1,character_event,10310849,168055,1.6299
2,standard,2336423,35639,1.5254
3,weapon_event,2839412,52821,1.8603



===== 저장 확인 =====
파일 존재: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/genshin_actual_rates.csv


In [ ]:

# 58단계: 원본 전체 데이터의 배너별 5성 획득 비율 계산

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# 1. 변조되지 않은 전체 원본 데이터
source_path = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2/"
    "genshin_processed/genshin_analysis_ready.parquet"
)

assert source_path.exists(), "원본 파일을 찾을 수 없습니다."

# 2. 기원 코드와 배너 유형 연결
banner_map = {
    "301": "character_event",
    "400": "character_event",
    "302": "weapon_event",
    "200": "standard",
    "100": "beginner"
}

# 3. 필요한 두 컬럼만 묶음 단위로 읽기
parquet_file = pq.ParquetFile(source_path)
result_parts = []
total_rows = 0

for i in range(parquet_file.num_row_groups):
    batch = parquet_file.read_row_group(
        i, columns=["gacha_type", "rank_type"]
    ).to_pandas()

    total_rows += len(batch)

    batch["banner_group"] = (
        batch["gacha_type"].astype(str).map(banner_map)
    )

    rank = pd.to_numeric(batch["rank_type"], errors="coerce")
    batch["is_5star"] = rank.eq(5).astype("int8")

    part = batch.groupby(
        "banner_group", dropna=False
    ).agg(
        total_pulls=("is_5star", "size"),
        five_star_count=("is_5star", "sum")
    ).reset_index()

    result_parts.append(part)

    if (i + 1) % 10 == 0 or i + 1 == parquet_file.num_row_groups:
        print(f"읽기 완료: {i + 1}/{parquet_file.num_row_groups}개 묶음")

# 4. 전체 집계
combined = pd.concat(result_parts, ignore_index=True)

actual_rates_df = (
    combined.groupby("banner_group", dropna=False)
    [["total_pulls", "five_star_count"]]
    .sum()
    .reset_index()
)

actual_rates_df["actual_rate"] = (
    actual_rates_df["five_star_count"]
    / actual_rates_df["total_pulls"]
)

# 5. 결과 저장
save_path = (
    Path("/content/drive/MyDrive/캡스톤 2/")
    / "capstonedata2/rf_model_data/genshin_actual_rates.csv"
)

actual_rates_df.to_csv(
    save_path, index=False, encoding="utf-8-sig"
)

# 6. 결과 확인
print("\n===== 58단계: 원본 전체 분석 결과 =====")
print("전체 원본 기록:", f"{total_rows:,}")
print("배너별 기록 합계:",
      f"{actual_rates_df['total_pulls'].sum():,}")

print("\n===== 배너별 실제 5성 획득 비율 =====")
display(
    actual_rates_df.assign(
        actual_rate=lambda x: (x["actual_rate"] * 100).round(4)
    ).rename(columns={
        "total_pulls": "뽑기 횟수",
        "five_star_count": "5성 획득 횟수",
        "actual_rate": "실제 획득 비율(%)"
    })
)

print("\n===== 저장 확인 =====")
print("파일 존재:", save_path.exists())
print("저장 경로:", save_path)


읽기 완료: 10/60개 묶음
읽기 완료: 20/60개 묶음
읽기 완료: 30/60개 묶음
읽기 완료: 40/60개 묶음
읽기 완료: 50/60개 묶음
읽기 완료: 60/60개 묶음

===== 58단계: 원본 전체 분석 결과 =====
전체 원본 기록: 15,598,804
배너별 기록 합계: 15,598,804

===== 배너별 실제 5성 획득 비율 =====


,banner_group,뽑기 횟수,5성 획득 횟수,실제 획득 비율(%)
0,beginner,112120,781,0.6966
1,character_event,10310849,168055,1.6299
2,standard,2336423,35639,1.5254
3,weapon_event,2839412,52821,1.8603



===== 저장 확인 =====
파일 존재: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/genshin_actual_rates.csv


In [ ]:

# 59단계: 공식 종합 확률과 실제 5성 획득 비율 비교

import pandas as pd
from pathlib import Path

# 1. 공식 확률과 실제 확률 연결
comparison_df = official_rates_df.merge(
    actual_rates_df,
    on="banner_group",
    how="left",
    validate="one_to_one"
)

# 2. 자료 연결 검사
assert comparison_df["total_pulls"].notna().all(), \
    "일부 기원의 실제 기록이 연결되지 않았습니다."

assert (comparison_df["total_pulls"] > 0).all(), \
    "뽑기 기록이 없는 기원이 있습니다."

# 3. 공식 확률과 실제 확률 차이
comparison_df["difference_pp"] = (
    comparison_df["actual_rate"]
    - comparison_df["consolidated_rate"]
) * 100

# 4. 표시용 표
report_df = comparison_df[[
    "banner_name",
    "total_pulls",
    "five_star_count",
    "consolidated_rate",
    "actual_rate",
    "difference_pp"
]].copy()

report_df["consolidated_rate"] *= 100
report_df["actual_rate"] *= 100

report_df = report_df.rename(columns={
    "banner_name": "기원 유형",
    "total_pulls": "뽑기 횟수",
    "five_star_count": "5성 획득 횟수",
    "consolidated_rate": "공식 종합 확률(%)",
    "actual_rate": "실제 획득 비율(%)",
    "difference_pp": "차이(%p)"
})

print("===== 59단계: 공식 확률 비교 =====")
display(report_df.round(4))

# 5. 분석 결과 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "genshin_probability_comparison.csv"
)

comparison_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== 결과 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("비교한 기원 유형:", len(comparison_df))
print(
    "분석 대상 뽑기 합계:",
    f"{int(comparison_df['total_pulls'].sum()):,}"
)


===== 59단계: 공식 확률 비교 =====


,기원 유형,뽑기 횟수,5성 획득 횟수,공식 종합 확률(%),실제 획득 비율(%),차이(%p)
0,캐릭터 이벤트 기원,10310849,168055,1.60,1.6299,0.0299
1,무기 이벤트 기원,2839412,52821,1.85,1.8603,0.0103
2,상시 기원,2336423,35639,1.60,1.5254,-0.0746



===== 결과 저장 확인 =====
저장 성공: True
비교한 기원 유형: 3
분석 대상 뽑기 합계: 15,486,684


In [ ]:

# 60단계: 실제 5성 획득 비율의 95% 신뢰구간 계산

import pandas as pd
from pathlib import Path
from statsmodels.stats.proportion import proportion_confint

# 1. 59단계 비교 데이터 복사
ci_df = comparison_df.copy()

# 2. Wilson 방식의 95% 신뢰구간 계산
ci_lower, ci_upper = proportion_confint(
    count=ci_df["five_star_count"].to_numpy(),
    nobs=ci_df["total_pulls"].to_numpy(),
    alpha=0.05,
    method="wilson"
)

ci_df["ci_lower"] = ci_lower
ci_df["ci_upper"] = ci_upper

# 3. 공식 종합 확률이 신뢰구간 안에 있는지 검사
ci_df["official_in_ci"] = (
    (ci_df["consolidated_rate"] >= ci_df["ci_lower"])
    & (ci_df["consolidated_rate"] <= ci_df["ci_upper"])
)

# 4. 보기 쉬운 결과표 생성
report_ci = pd.DataFrame({
    "기원 유형": ci_df["banner_name"],
    "공식 종합 확률(%)": ci_df["consolidated_rate"] * 100,
    "실제 획득 비율(%)": ci_df["actual_rate"] * 100,
    "95% 하한(%)": ci_df["ci_lower"] * 100,
    "95% 상한(%)": ci_df["ci_upper"] * 100,
    "공식 확률 구간 포함": ci_df["official_in_ci"].map({
        True: "포함",
        False: "미포함"
    })
})

print("===== 60단계: 95% 신뢰구간 =====")
display(report_ci.round(4))

# 5. 분석 결과 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "genshin_probability_ci95.csv"
)

ci_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== 저장 및 검증 =====")
print("저장 성공:", save_path.exists())
print("분석 기원 유형:", len(ci_df))
print("신뢰구간 누락:", ci_df[["ci_lower", "ci_upper"]].isna().sum().sum())


===== 60단계: 95% 신뢰구간 =====


,기원 유형,공식 종합 확률(%),실제 획득 비율(%),95% 하한(%),95% 상한(%),공식 확률 구간 포함
0,캐릭터 이벤트 기원,1.60,1.6299,1.6222,1.6376,미포함
1,무기 이벤트 기원,1.85,1.8603,1.8446,1.8761,포함
2,상시 기원,1.60,1.5254,1.5097,1.5412,미포함



===== 저장 및 검증 =====
저장 성공: True
분석 기원 유형: 3
신뢰구간 누락: 0


In [ ]:

# 61단계: 공식 종합 확률에 대한 기초 이항검정

import pandas as pd
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# 1. 60단계 결과 복사
test_df = ci_df.copy()

# 2. 배너별 양측 이항검정
p_values = []

for _, row in test_df.iterrows():
    result = binomtest(
        k=int(row["five_star_count"]),
        n=int(row["total_pulls"]),
        p=float(row["consolidated_rate"]),
        alternative="two-sided"
    )
    p_values.append(result.pvalue)

test_df["p_value"] = p_values

# 3. 세 가지 배너를 동시에 검정하므로
#    Holm 방식으로 다중검정 보정
test_df["p_adjusted"] = multipletests(
    test_df["p_value"],
    alpha=0.05,
    method="holm"
)[1]

test_df["significant_05"] = (
    test_df["p_adjusted"] < 0.05
)

# 4. 결과표
report_test = pd.DataFrame({
    "기원 유형": test_df["banner_name"],
    "공식 확률(%)": test_df["consolidated_rate"] * 100,
    "관측 비율(%)": test_df["actual_rate"] * 100,
    "차이(%p)": test_df["difference_pp"],
    "p값": test_df["p_value"],
    "보정 p값": test_df["p_adjusted"],
    "통계적 차이": test_df["significant_05"].map({
        True: "있음",
        False: "확인되지 않음"
    })
})

print("===== 61단계: 기초 이항검정 결과 =====")
display(report_test.round(6))

print("\n===== 검증 =====")
print("검정한 기원 유형:", len(test_df))
print("p값 결측:", test_df["p_value"].isna().sum())
print("유의수준: 0.05")
print("다중검정 보정: Holm")

# 5. 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "genshin_probability_statistical_test.csv"
)

test_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== 저장 확인 =====")
print("파일 존재:", save_path.exists())


===== 61단계: 기초 이항검정 결과 =====


,기원 유형,공식 확률(%),관측 비율(%),차이(%p),p값,보정 p값,통계적 차이
0,캐릭터 이벤트 기원,1.60,1.629885,0.029885,0.000000,0.000000,있음
1,무기 이벤트 기원,1.85,1.860280,0.010280,0.198446,0.198446,확인되지 않음
2,상시 기원,1.60,1.525366,-0.074634,0.000000,0.000000,있음



===== 검증 =====
검정한 기원 유형: 3
p값 결측: 0
유의수준: 0.05
다중검정 보정: Holm

===== 저장 확인 =====
파일 존재: True


In [ ]:

# 62단계: 사용자 단위 부트스트랩 신뢰구간

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

source_path = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2/"
    "genshin_processed/genshin_analysis_ready.parquet"
)

banner_map = {
    "301": "character_event",
    "400": "character_event",
    "302": "weapon_event",
    "200": "standard"
}

# 1. 원본 전체 기록을 사용자 + 배너별로 집계
pf = pq.ParquetFile(source_path)
parts = []

for i in range(pf.num_row_groups):
    batch = pf.read_row_group(
        i, columns=["user_id", "gacha_type", "rank_type"]
    ).to_pandas()

    batch["banner_group"] = (
        batch["gacha_type"].astype(str).map(banner_map)
    )

    batch = batch.dropna(subset=["banner_group"])

    batch["five_star"] = (
        pd.to_numeric(batch["rank_type"], errors="coerce")
        .eq(5).astype("int8")
    )

    part = batch.groupby(
        ["user_id", "banner_group"], as_index=False
    ).agg(
        pulls=("five_star", "size"),
        fives=("five_star", "sum")
    )

    parts.append(part)

    if (i + 1) % 10 == 0:
        print(f"사용자별 집계: {i + 1}/{pf.num_row_groups}")

user_banner_df = pd.concat(parts).groupby(
    ["user_id", "banner_group"], as_index=False
)[["pulls", "fives"]].sum()

# 2. 모든 사용자와 배너를 같은 순서로 배열
users = sorted(user_banner_df["user_id"].unique())
banners = ["character_event", "weapon_event", "standard"]

pull_matrix = user_banner_df.pivot(
    index="user_id", columns="banner_group", values="pulls"
).reindex(index=users, columns=banners).fillna(0).to_numpy()

five_matrix = user_banner_df.pivot(
    index="user_id", columns="banner_group", values="fives"
).reindex(index=users, columns=banners).fillna(0).to_numpy()

# 3. 사용자 단위 재표본추출 1,000회
rng = np.random.default_rng(42)
bootstrap_rates = []

for _ in range(1000):
    idx = rng.integers(0, len(users), size=len(users))

    total_pulls = pull_matrix[idx].sum(axis=0)
    total_fives = five_matrix[idx].sum(axis=0)

    rates = np.divide(
        total_fives,
        total_pulls,
        out=np.full(len(banners), np.nan),
        where=total_pulls > 0
    )
    bootstrap_rates.append(rates)

bootstrap_rates = np.array(bootstrap_rates)

# 4. 95% 신뢰구간
ci_low = np.nanpercentile(bootstrap_rates, 2.5, axis=0)
ci_high = np.nanpercentile(bootstrap_rates, 97.5, axis=0)

official_map = dict(zip(
    official_rates_df["banner_group"],
    official_rates_df["consolidated_rate"]
))

results = []

for j, banner in enumerate(banners):
    actual = five_matrix[:, j].sum() / pull_matrix[:, j].sum()
    official = official_map[banner]

    results.append({
        "기원 유형": banner,
        "공식 확률(%)": official * 100,
        "실제 비율(%)": actual * 100,
        "사용자 단위 95% 하한(%)": ci_low[j] * 100,
        "사용자 단위 95% 상한(%)": ci_high[j] * 100,
        "공식 확률 포함": (
            ci_low[j] <= official <= ci_high[j]
        )
    })

bootstrap_result_df = pd.DataFrame(results)

print("\n===== 62단계: 사용자 단위 신뢰구간 =====")
print("분석 사용자 수:", len(users))
print("부트스트랩 반복:", len(bootstrap_rates))

display(bootstrap_result_df.round(4))

# 5. CSV 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2/"
    "rf_model_data/genshin_user_bootstrap_ci.csv"
)

bootstrap_result_df.to_csv(
    save_path, index=False, encoding="utf-8-sig"
)

print("\n저장 성공:", save_path.exists())


사용자별 집계: 10/60
사용자별 집계: 20/60
사용자별 집계: 30/60
사용자별 집계: 40/60
사용자별 집계: 50/60
사용자별 집계: 60/60

===== 62단계: 사용자 단위 신뢰구간 =====
분석 사용자 수: 16289
부트스트랩 반복: 1000


,기원 유형,공식 확률(%),실제 비율(%),사용자 단위 95% 하한(%),사용자 단위 95% 상한(%),공식 확률 포함
0,character_event,1.60,1.6299,1.6264,1.6334,False
1,weapon_event,1.85,1.8603,1.8535,1.8668,False
2,standard,1.60,1.5254,1.5170,1.5336,False



저장 성공: True


In [ ]:

# 63단계: 신뢰구간 변화 원인 분석

import numpy as np
import pandas as pd

banners = ["character_event", "weapon_event", "standard"]

rows = []

for j, banner in enumerate(banners):
    # 해당 기원을 이용한 사용자만 계산
    active_mask = pull_matrix[:, j] > 0
    active_pulls = pull_matrix[active_mask, j]

    total_pulls = int(pull_matrix[:, j].sum())
    total_fives = int(five_matrix[:, j].sum())
    actual_rate = total_fives / total_pulls

    # 독립 시행을 가정한 표준오차
    binomial_se = np.sqrt(
        actual_rate * (1 - actual_rate) / total_pulls
    )

    # 사용자 재표본추출 결과의 표준오차
    bootstrap_se = np.std(
        bootstrap_rates[:, j], ddof=1
    )

    rows.append({
        "기원": banner,
        "이용 사용자": int(active_mask.sum()),
        "사용자당 중앙 뽑기": float(np.median(active_pulls)),
        "사용자당 최대 뽑기": int(active_pulls.max()),
        "단순 표준오차(%p)": binomial_se * 100,
        "부트스트랩 표준오차(%p)": bootstrap_se * 100,
        "표준오차 비율": bootstrap_se / binomial_se,
        "단순 구간 폭(%p)": float(
            (ci_df.set_index("banner_group")
             .loc[banner, "ci_upper"]
             - ci_df.set_index("banner_group")
             .loc[banner, "ci_lower"]) * 100
        ),
        "사용자 구간 폭(%p)": float(
            (ci_high[j] - ci_low[j]) * 100
        )
    })

diagnostic_df = pd.DataFrame(rows)

print("===== 63단계: 신뢰구간 진단 =====")
display(diagnostic_df.round(5))

print("\n===== 해석 기준 =====")
print("표준오차 비율 < 1: 부트스트랩 변동이 더 작음")
print("표준오차 비율 > 1: 부트스트랩 변동이 더 큼")
print("주의: 어느 방법이 더 좁다고 더 정확한 것은 아님")

print("\n===== 데이터 확인 =====")
print("전체 사용자:", len(users))
print("부트스트랩 반복:", len(bootstrap_rates))


===== 63단계: 신뢰구간 진단 =====


,기원,이용 사용자,사용자당 중앙 뽑기,사용자당 최대 뽑기,단순 표준오차(%p),부트스트랩 표준오차(%p),표준오차 비율,단순 구간 폭(%p),사용자 구간 폭(%p)
0,character_event,16277,496.0,12160,0.00394,0.00179,0.45272,0.01546,0.00703
1,weapon_event,14258,116.0,22940,0.00802,0.00364,0.45381,0.03143,0.01333
2,standard,16245,132.0,3331,0.00802,0.00421,0.52520,0.03143,0.01664



===== 해석 기준 =====
표준오차 비율 < 1: 부트스트랩 변동이 더 작음
표준오차 비율 > 1: 부트스트랩 변동이 더 큼
주의: 어느 방법이 더 좁다고 더 정확한 것은 아님

===== 데이터 확인 =====
전체 사용자: 16289
부트스트랩 반복: 1000


In [ ]:

# 64단계: 사용자별 기록량에 따른 획득 비율 분석

import pandas as pd
import numpy as np

# 62단계에서 만든 사용자별 집계 데이터 이용
thresholds = [1, 20, 100, 500]
rows = []

official_map = dict(zip(
    official_rates_df["banner_group"],
    official_rates_df["consolidated_rate"]
))

for banner in ["character_event", "weapon_event", "standard"]:
    data = user_banner_df.loc[
        user_banner_df["banner_group"] == banner
    ].copy()

    for threshold in thresholds:
        selected = data.loc[
            data["pulls"] >= threshold
        ]

        if len(selected) == 0:
            continue

        # 기록 수에 따라 가중되는 전체 획득 비율
        pooled_rate = (
            selected["fives"].sum()
            / selected["pulls"].sum()
        )

        # 사용자별 획득 비율을 동일 가중치로 평균
        user_rates = selected["fives"] / selected["pulls"]
        user_mean_rate = user_rates.mean()

        rows.append({
            "기원": banner,
            "최소 뽑기 횟수": threshold,
            "사용자 수": len(selected),
            "총 뽑기 횟수": int(selected["pulls"].sum()),
            "공식 확률(%)": official_map[banner] * 100,
            "전체 획득 비율(%)": pooled_rate * 100,
            "사용자 평균 비율(%)": user_mean_rate * 100
        })

sensitivity_df = pd.DataFrame(rows)

print("===== 64단계: 사용자 기록량별 확률 비교 =====")
display(sensitivity_df.round(4))

print("\n===== 분석 기준 =====")
print("전체 획득 비율: 모든 뽑기 기록을 합쳐 계산")
print("사용자 평균 비율: 각 이용자의 비율을 동일하게 반영")

print("\n===== 원본 데이터 확인 =====")
print("분석 사용자:", user_banner_df["user_id"].nunique())
print("분석 기원 유형:", user_banner_df["banner_group"].nunique())


===== 64단계: 사용자 기록량별 확률 비교 =====


,기원,최소 뽑기 횟수,사용자 수,총 뽑기 횟수,공식 확률(%),전체 획득 비율(%),사용자 평균 비율(%)
0,character_event,1,16277,10310849,1.60,1.6299,1.6396
1,character_event,20,16241,10310480,1.60,1.6299,1.6383
2,character_event,100,15729,10276687,1.60,1.6299,1.6393
3,character_event,500,8083,7787748,1.60,1.6242,1.6266
4,weapon_event,1,14258,2839412,1.85,1.8603,1.7346
5,weapon_event,20,12280,2824078,1.85,1.8629,1.7894
6,weapon_event,100,7746,2564630,1.85,1.8811,1.8854
7,weapon_event,500,1058,1067768,1.85,1.8750,1.8689
8,standard,1,16245,2336423,1.60,1.5254,1.5026
9,standard,20,16060,2334345,1.60,1.5254,1.5041



===== 분석 기준 =====
전체 획득 비율: 모든 뽑기 기록을 합쳐 계산
사용자 평균 비율: 각 이용자의 비율을 동일하게 반영

===== 원본 데이터 확인 =====
분석 사용자: 16289
분석 기원 유형: 3


In [ ]:

# 65단계: 뽑기 기록의 사용자 편중도 검사

import numpy as np
import pandas as pd

result_rows = []

for banner in ["character_event", "weapon_event", "standard"]:
    data = user_banner_df.loc[
        user_banner_df["banner_group"] == banner
    ].copy()

    data = data.sort_values("pulls", ascending=False)

    # 상위 1% 사용자 수 (최소 1명)
    top_n = max(1, int(np.ceil(len(data) * 0.01)))

    top_users = data.head(top_n)

    total_pulls = int(data["pulls"].sum())
    top_pulls = int(top_users["pulls"].sum())

    result_rows.append({
        "기원": banner,
        "전체 사용자": len(data),
        "상위 1% 사용자": top_n,
        "전체 뽑기": total_pulls,
        "상위 1% 뽑기": top_pulls,
        "상위 1% 기록 비중(%)": (
            top_pulls / total_pulls * 100
        ),
        "5성 획득 0건 사용자": int(
            (data["fives"] == 0).sum()
        ),
        "사용자당 중앙 뽑기": float(
            data["pulls"].median()
        )
    })

concentration_df = pd.DataFrame(result_rows)

print("===== 65단계: 사용자 기록 편중도 =====")
display(concentration_df.round(2))

print("\n===== 집계 검증 =====")
print(
    "세 기원 뽑기 합계:",
    f"{concentration_df['전체 뽑기'].sum():,}"
)

print("\n===== 분석 기준 =====")
print("상위 1%: 각 기원에서 뽑기 횟수가 가장 많은 사용자")
print("기록 비중: 이 사용자들이 차지하는 전체 뽑기 비율")
print("※ 비중이 높으면 일부 사용자에게 결과가 집중될 수 있음")


===== 65단계: 사용자 기록 편중도 =====


,기원,전체 사용자,상위 1% 사용자,전체 뽑기,상위 1% 뽑기,상위 1% 기록 비중(%),5성 획득 0건 사용자,사용자당 중앙 뽑기
0,character_event,16277,163,10310849,752885,7.30,163,496.0
1,weapon_event,14258,143,2839412,379141,13.35,2974,116.0
2,standard,16245,163,2336423,91490,3.92,965,132.0



===== 집계 검증 =====
세 기원 뽑기 합계: 15,486,684

===== 분석 기준 =====
상위 1%: 각 기원에서 뽑기 횟수가 가장 많은 사용자
기록 비중: 이 사용자들이 차지하는 전체 뽑기 비율
※ 비중이 높으면 일부 사용자에게 결과가 집중될 수 있음


In [ ]:

# 66단계: 두 종류의 합성 변조 기록 통합

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

# 1. 두 변조 데이터 준비
rank_changes = tampered_df.copy()
rank_changes["tamper_type"] = "item_and_rank_change"

character_changes = tampered_char_df.copy()
character_changes["tamper_type"] = "character_name_change"

# 2. 변조 기록 1,500건 합치기
combined_tampered_df = pd.concat(
    [rank_changes, character_changes],
    ignore_index=True
)

assert len(combined_tampered_df) == 1500
assert not combined_tampered_df.duplicated(KEY_COLS).any()

# 3. 정상 원본 복사
combined_history_df = normal_df.copy()

combined_history_df = combined_history_df.set_index(
    KEY_COLS, drop=False
)

replacement = combined_tampered_df.set_index(
    KEY_COLS, drop=False
)

assert replacement.index.isin(
    combined_history_df.index
).all()

# 4. 실제로 변경한 컬럼만 교체
replace_cols = [
    "name_zh", "name_ko",
    "rank_type", "item_type", "item_type_ko",
    "is_five_star", "is_four_star"
]

for col in replace_cols:
    target_dtype = combined_history_df[col].dtype
    values = replacement[col].astype(target_dtype)

    combined_history_df.loc[
        replacement.index, col
    ] = values.to_numpy()

# 5. 정답 라벨 적용
combined_history_df["label"] = 0
combined_history_df["tamper_type"] = "normal"

combined_history_df.loc[
    replacement.index, "label"
] = 1

combined_history_df.loc[
    replacement.index, "tamper_type"
] = replacement["tamper_type"].to_numpy()

combined_history_df = combined_history_df.reset_index(
    drop=True
)

# 6. 검사
print("===== 66단계: 통합 결과 =====")
print("전체 기록:", len(combined_history_df))
print("사용자 수:", combined_history_df["user_id"].nunique())

print("\n===== 정상·변조 기록 =====")
print(combined_history_df["label"].value_counts().sort_index())

print("\n===== 변조 유형별 기록 =====")
print(combined_history_df["tamper_type"].value_counts())

print("\n===== 검증 =====")
print("중복 식별자:",
      combined_history_df.duplicated(KEY_COLS).sum())

print("5성 표시 불일치:",
      (combined_history_df["is_five_star"]
       != (combined_history_df["rank_type"] == 5)).sum())

print("4성 표시 불일치:",
      (combined_history_df["is_four_star"]
       != (combined_history_df["rank_type"] == 4)).sum())

print("원본 정상 후보 유지:", len(normal_df))


===== 66단계: 통합 결과 =====
전체 기록: 99998
사용자 수: 116

===== 정상·변조 기록 =====
label
0    98498
1     1500
Name: count, dtype: int64

===== 변조 유형별 기록 =====
tamper_type
normal                   98498
item_and_rank_change      1000
character_name_change      500
Name: count, dtype: int64

===== 검증 =====
중복 식별자: 0
5성 표시 불일치: 0
4성 표시 불일치: 0
원본 정상 후보 유지: 99998


In [ ]:

# 67단계: 통합 합성 변조 이력의 천장 변수 재계산

import pandas as pd
import numpy as np

# 1. 통합 데이터 복사
rf_history_df = combined_history_df.copy()

group_cols = ["user_id", "banner_group"]

# 2. 사용자별, 기원별, 시간순 정렬
rf_history_df = rf_history_df.sort_values(
    [
        "user_id",
        "banner_group",
        "gacha_time",
        "gacha_id_num"
    ],
    kind="stable"
).reset_index(drop=True)

# 3. 그룹 내 뽑기 순서
rf_history_df["group_order"] = (
    rf_history_df.groupby(group_cols).cumcount()
)

# 4. 5성 획득 위치
five_positions = rf_history_df["group_order"].where(
    rf_history_df["rank_type"] == 5
)

# 5. 이전 또는 현재의 가장 최근 5성 위치
last_five = five_positions.groupby(
    [
        rf_history_df["user_id"],
        rf_history_df["banner_group"]
    ]
).ffill()

# 6. 마지막 5성 이후 누적 횟수
rf_history_df["pulls_since_last_5star"] = (
    rf_history_df["group_order"] - last_five
)

# 7. 현재 뽑기 직전의 천장 상태
rf_history_df["pity_before_pull"] = (
    rf_history_df.groupby(group_cols)[
        "pulls_since_last_5star"
    ].shift(1)
)

# 8. 현재 뽑기 포함 횟수
rf_history_df["current_pity_count"] = (
    rf_history_df["pity_before_pull"] + 1
)

# 9. 천장 기준 및 위반 여부
pity_limits = {
    "character_event": 90,
    "weapon_event": 80,
    "standard": 90
}

rf_history_df["pity_limit"] = (
    rf_history_df["banner_group"].map(pity_limits)
)

rf_history_df["pity_rule_violation"] = (
    rf_history_df["current_pity_count"]
    > rf_history_df["pity_limit"]
).fillna(False)

# 10. 검증
five_reset_errors = (
    (rf_history_df["rank_type"] == 5)
    & rf_history_df["pulls_since_last_5star"].notna()
    & (rf_history_df["pulls_since_last_5star"] != 0)
).sum()

print("===== 67단계: 천장 변수 재계산 =====")
print("전체 기록:", len(rf_history_df))
print("정상 후보:", (rf_history_df["label"] == 0).sum())
print("합성 변조:", (rf_history_df["label"] == 1).sum())

print("\n===== 천장 상태 검사 =====")
print("5성 기록:", (rf_history_df["rank_type"] == 5).sum())
print("5성 초기화 오류:", five_reset_errors)
print("천장 초과 의심 기록:",
      rf_history_df["pity_rule_violation"].sum())

print("\n===== 변조 유형별 분포 =====")
print(rf_history_df["tamper_type"].value_counts())

print("\n===== 결측값 확인 =====")
print("직전 천장 상태 미확인:",
      rf_history_df["pity_before_pull"].isna().sum())


===== 67단계: 천장 변수 재계산 =====
전체 기록: 99998
정상 후보: 98498
합성 변조: 1500

===== 천장 상태 검사 =====
5성 기록: 2649
5성 초기화 오류: 0
천장 초과 의심 기록: 0

===== 변조 유형별 분포 =====
tamper_type
normal                   98498
item_and_rank_change      1000
character_name_change      500
Name: count, dtype: int64

===== 결측값 확인 =====
직전 천장 상태 미확인: 12287


In [ ]:

# 68단계: 랜덤 포레스트 학습 전 변수 점검

import pandas as pd

# 현재 기록만으로 계산한 정답성 변수는 제외
excluded_features = [
    "user_id",
    "gacha_id",
    "gacha_id_num",
    "name_zh",
    "name_ko",
    "label",
    "tamper_type",
    "is_five_star",
    "is_four_star",
    "pulls_since_last_5star",
    "current_pity_count",
    "pity_rule_violation"
]

# 우선 검토할 모델 입력 후보
candidate_features = [
    "gacha_type",
    "item_type",
    "rank_type",
    "time_diff_seconds",
    "pity_before_pull",
    "pull_order",
    "banner_pull_order"
]

rows = []

for col in candidate_features:
    rows.append({
        "변수": col,
        "자료형": str(rf_history_df[col].dtype),
        "결측값": int(rf_history_df[col].isna().sum()),
        "고유값 수": int(rf_history_df[col].nunique())
    })

feature_check_df = pd.DataFrame(rows)

print("===== 68단계: 모델 입력 후보 점검 =====")
display(feature_check_df)

print("\n===== 변조 유형별 등급 분포 =====")
display(
    pd.crosstab(
        rf_history_df["tamper_type"],
        rf_history_df["rank_type"]
    )
)

print("\n===== 검증 =====")
print("전체 기록:", len(rf_history_df))
print("합성 변조 기록:", int(rf_history_df["label"].sum()))
print("모델 후보 변수:", len(candidate_features))
print("입력 후보 중 제외 변수 중복:",
      len(set(candidate_features) & set(excluded_features)))


===== 68단계: 모델 입력 후보 점검 =====


,변수,자료형,결측값,고유값 수
0,gacha_type,object,0,5
1,item_type,int8,0,2
2,rank_type,int8,0,3
3,time_diff_seconds,float64,383,17183
4,pity_before_pull,float64,12287,84
5,pull_order,int64,0,3856
6,banner_pull_order,int64,0,2134



===== 변조 유형별 등급 분포 =====


rank_type,3,4,5
tamper_type,,,
character_name_change,0,500,0
item_and_rank_change,0,0,1000
normal,84036,12813,1649



===== 검증 =====
전체 기록: 99998
합성 변조 기록: 1500
모델 후보 변수: 7
입력 후보 중 제외 변수 중복: 0


In [ ]:

# 69단계: 변조 전후 모델 입력 변수 비교

import pandas as pd

KEY_COLS = ["user_id", "gacha_id"]

candidate_features = [
    "gacha_type",
    "item_type",
    "rank_type",
    "time_diff_seconds",
    "pity_before_pull",
    "pull_order",
    "banner_pull_order"
]

# 원본 기록과 합성 변조 기록 연결
original_rows = normal_df.set_index(KEY_COLS)
modified_rows = rf_history_df.loc[
    rf_history_df["label"] == 1
].set_index(KEY_COLS)

original_matched = original_rows.loc[
    modified_rows.index, candidate_features
]

modified_matched = modified_rows[candidate_features]

# 결측값끼리는 동일한 것으로 처리
comparison = (
    original_matched.eq(modified_matched)
    | (
        original_matched.isna()
        & modified_matched.isna()
    )
)

changed = ~comparison

# 변조 유형별 변경 변수 집계
rows = []

for tamper_type, group in modified_rows.groupby("tamper_type"):
    indexes = group.index
    type_changes = changed.loc[indexes]

    result = {
        "변조 유형": tamper_type,
        "기록 수": len(group),
        "입력 변수 변화 0개": int(
            (~type_changes.any(axis=1)).sum()
        ),
        "입력 변수 변화 1개 이상": int(
            type_changes.any(axis=1).sum()
        )
    }

    for feature in candidate_features:
        result[feature] = int(
            type_changes[feature].sum()
        )

    rows.append(result)

change_report_df = pd.DataFrame(rows)

print("===== 69단계: 변조 전후 입력 변수 비교 =====")
display(change_report_df)

print("\n===== 검사 대상 =====")
print("합성 변조 기록:", len(modified_rows))
print("후보 변수 수:", len(candidate_features))
print("원본 기록 연결 성공:", len(original_matched))


===== 69단계: 변조 전후 입력 변수 비교 =====


,변조 유형,기록 수,입력 변수 변화 0개,입력 변수 변화 1개 이상,gacha_type,item_type,rank_type,time_diff_seconds,pity_before_pull,pull_order,banner_pull_order
0,character_name_change,500,370,130,0,0,0,0,130,0,0
1,item_and_rank_change,1000,0,1000,0,0,1000,0,280,0,0



===== 검사 대상 =====
합성 변조 기록: 1500
후보 변수 수: 7
원본 기록 연결 성공: 1500


In [ ]:

# 70단계: 캐릭터 이름 변경의 탐지 단서 분석

import pandas as pd

keys = ["user_id", "gacha_id"]

original = normal_df.set_index(keys)

changed_chars = (
    rf_history_df.loc[
        rf_history_df["tamper_type"]
        == "character_name_change"
    ]
    .copy()
    .set_index(keys)
)

# 원본 이름 연결
changed_chars["original_name"] = (
    original.loc[changed_chars.index, "name_ko"]
    .to_numpy()
)

# 이름 변경 확인
changed_chars["name_changed"] = (
    changed_chars["original_name"]
    != changed_chars["name_ko"]
)

# 학습 사용자 원본에서 캐릭터별 최초 관측 시점 계산
train_first_seen = (
    normal_df.loc[
        (normal_df["rank_type"] == 4)
        & (normal_df["item_type_ko"] == "캐릭터")
    ]
    .groupby("name_ko")["gacha_time"]
    .min()
)

# 최초 관측 시점과 변조 기록 비교
changed_chars["first_seen"] = (
    changed_chars["name_ko"].map(train_first_seen)
)

changed_chars["before_first_seen"] = (
    changed_chars["gacha_time"]
    < changed_chars["first_seen"]
)

print("===== 70단계: 캐릭터 변조 탐지 단서 =====")
print("검사 기록:", len(changed_chars))
print("캐릭터 이름 변경:",
      int(changed_chars["name_changed"].sum()))
print("최초 관측 이전:",
      int(changed_chars["before_first_seen"].sum()))
print("최초 관측일 정보 없음:",
      int(changed_chars["first_seen"].isna().sum()))

print("\n===== 최초 관측 이전 캐릭터별 기록 =====")
display(
    changed_chars.loc[
        changed_chars["before_first_seen"]
    ]["name_ko"].value_counts().rename_axis(
        "캐릭터"
    ).reset_index(name="의심 기록 수")
)

print("\n===== 현재 데이터 확인 =====")
print("전체 이력:", len(rf_history_df))
print("합성 변조:", int(rf_history_df["label"].sum()))


===== 70단계: 캐릭터 변조 탐지 단서 =====
검사 기록: 500
캐릭터 이름 변경: 500
최초 관측 이전: 53
최초 관측일 정보 없음: 0

===== 최초 관측 이전 캐릭터별 기록 =====


,캐릭터,의심 기록 수
0,쿠키 시노부,20
1,시카노인 헤이조,15
2,운근,7
3,고로,4
4,토마,3
5,케이아,2
6,연비,1
7,엠버,1



===== 현재 데이터 확인 =====
전체 이력: 99998
합성 변조: 1500


In [ ]:

# 71단계: 정상 기록의 최초 관측일 기준 오탐 검사
# 검사 대상 사용자 자신의 기록을 제외한 기준 사용

import pandas as pd

# 1. 변조되지 않은 원본의 4성 캐릭터 기록
baseline_df = normal_df.loc[
    (normal_df["rank_type"] == 4) &
    (normal_df["item_type_ko"] == "캐릭터"),
    ["user_id", "name_ko", "gacha_time"]
].copy()

# 2. 캐릭터별 가장 이른 관측 기록 2개를
#    서로 다른 사용자 기준으로 찾기
first_by_user = (
    baseline_df.groupby(
        ["name_ko", "user_id"], as_index=False
    )["gacha_time"].min()
    .sort_values(["name_ko", "gacha_time"])
)

first_two = (
    first_by_user.groupby("name_ko")
    .head(2)
    .copy()
)

first_two["position"] = (
    first_two.groupby("name_ko").cumcount()
)

first_dates = first_two.pivot(
    index="name_ko",
    columns="position",
    values="gacha_time"
)

first_users = first_two.pivot(
    index="name_ko",
    columns="position",
    values="user_id"
)

# 3. 사용자 본인의 기록이 첫 관측인 경우
#    다른 사용자의 두 번째 관측일 사용
baseline_df["first_date"] = (
    baseline_df["name_ko"].map(first_dates[0])
)

baseline_df["first_user"] = (
    baseline_df["name_ko"].map(first_users[0])
)

baseline_df["second_date"] = (
    baseline_df["name_ko"].map(first_dates[1])
)

baseline_df["reference_date"] = (
    baseline_df["first_date"]
)

own_first = (
    baseline_df["user_id"] ==
    baseline_df["first_user"]
)

baseline_df.loc[
    own_first, "reference_date"
] = baseline_df.loc[
    own_first, "second_date"
]

# 4. 다른 사용자 관측일보다 이른 정상 기록
baseline_df["flagged"] = (
    baseline_df["gacha_time"]
    < baseline_df["reference_date"]
)

# 5. 결과
print("===== 71단계: 정상 기록 오탐 검사 =====")
print("정상 4성 캐릭터 기록:", len(baseline_df))
print("기준 날짜 없음:",
      baseline_df["reference_date"].isna().sum())
print("의심으로 표시된 정상 기록:",
      int(baseline_df["flagged"].sum()))

print("\n===== 캐릭터별 의심 기록 =====")
display(
    baseline_df.loc[baseline_df["flagged"], "name_ko"]
    .value_counts()
    .head(15)
)

print("\n===== 정상 기록의 오탐 비율 =====")
valid = baseline_df["reference_date"].notna()

print(
    "오탐 비율(%):",
    round(
        baseline_df.loc[valid, "flagged"].mean() * 100,
        4
    )
)


===== 71단계: 정상 기록 오탐 검사 =====
정상 4성 캐릭터 기록: 7797
기준 날짜 없음: 0
의심으로 표시된 정상 기록: 45

===== 캐릭터별 의심 기록 =====


,count
name_ko,
신염,5
연비,4
북두,4
쿠죠 사라,4
시카노인 헤이조,4
운근,2
리사,2
행추,2
향릉,2



===== 정상 기록의 오탐 비율 =====
오탐 비율(%): 0.5771


In [ ]:

# 72단계: 학습·검증·테스트 사용자 분리 상태 확인

import pandas as pd
from pathlib import Path

split_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/rf_split_users.csv"
)

assert split_path.exists(), "사용자 분리 파일이 없습니다."

split_users_df = pd.read_csv(
    split_path,
    dtype={"user_id": str}
)

print("===== 사용자 분리 파일 구조 =====")
print("컬럼:", split_users_df.columns.tolist())
print("전체 사용자:", len(split_users_df))

print("\n===== 분리 파일 예시 =====")
display(split_users_df.head())

print("\n===== 현재 실험 데이터 =====")
print("전체 기록:", len(rf_history_df))
print("사용자 수:", rf_history_df["user_id"].nunique())

print("\n===== 라벨 분포 =====")
display(
    pd.crosstab(
        rf_history_df["tamper_type"],
        rf_history_df["label"]
    )
)

print("\n===== 사용자 중복 검사 =====")
print(
    "실험 데이터 내부 중복 사용자 ID:",
    rf_history_df["user_id"].nunique()
)

print(
    "분리 파일 내부 중복 사용자 ID:",
    split_users_df["user_id"].duplicated().sum()
)


===== 사용자 분리 파일 구조 =====
컬럼: ['user_id', 'split']
전체 사용자: 16289

===== 분리 파일 예시 =====


,user_id,split
0,0003f01339643530,train
1,0008dd4b99fd4775,train
2,000c3dd8eb5d4961,train
3,00148ebafe2d0393,train
4,00173dcd62005005,train



===== 현재 실험 데이터 =====
전체 기록: 99998
사용자 수: 116

===== 라벨 분포 =====


label,0,1
tamper_type,,
character_name_change,0,500
item_and_rank_change,0,1000
normal,98498,0



===== 사용자 중복 검사 =====
실험 데이터 내부 중복 사용자 ID: 116
분리 파일 내부 중복 사용자 ID: 0


In [ ]:

# 73단계: 실험 데이터 사용자 분리 검증

import pandas as pd

# 1. 실험 데이터 사용자 목록
experiment_users = pd.DataFrame({
    "user_id": rf_history_df["user_id"].astype(str).unique()
})

# 2. 기존 분리 파일과 연결
experiment_user_splits = experiment_users.merge(
    split_users_df,
    on="user_id",
    how="left",
    validate="one_to_one"
)

# 3. 사용자별 분리 현황
print("===== 73단계: 실험 사용자 분리 확인 =====")
print("실험 사용자:", len(experiment_users))

print("\n===== 사용자별 소속 =====")
print(
    experiment_user_splits["split"]
    .value_counts(dropna=False)
)

# 4. 기록 단위 소속 확인
record_splits = rf_history_df.merge(
    split_users_df,
    on="user_id",
    how="left",
    validate="many_to_one"
)

print("\n===== 기록별 소속 =====")
print(
    pd.crosstab(
        record_splits["split"],
        record_splits["label"],
        dropna=False
    )
)

# 5. 누락 및 학습 사용자 외 포함 검사
missing_users = experiment_user_splits["split"].isna().sum()

non_train_users = (
    experiment_user_splits["split"].notna()
    & experiment_user_splits["split"].ne("train")
).sum()

print("\n===== 최종 검증 =====")
print("분리 정보 없는 사용자:", missing_users)
print("학습 그룹 외 사용자:", non_train_users)
print(
    "전체 기록 유지:",
    len(record_splits) == len(rf_history_df)
)


===== 73단계: 실험 사용자 분리 확인 =====
실험 사용자: 116

===== 사용자별 소속 =====
split
train    116
Name: count, dtype: int64

===== 기록별 소속 =====
label      0     1
split             
train  98498  1500

===== 최종 검증 =====
분리 정보 없는 사용자: 0
학습 그룹 외 사용자: 0
전체 기록 유지: True


In [2]:

# 현재 코랩에 필요한 변수가 존재하는지 확인

required_variables = [
    "normal_df",
    "tampered_df",
    "tampered_char_df",
    "combined_history_df",
    "rf_history_df",
    "split_users_df"
]

print("===== 코랩 변수 상태 확인 =====")

for name in required_variables:
    if name in globals():
        obj = globals()[name]
        print(f"[존재] {name}: {len(obj):,}건")
    else:
        print(f"[없음] {name}")

print("\n===== 검사 완료 =====")


===== 코랩 변수 상태 확인 =====
[없음] normal_df
[없음] tampered_df
[없음] tampered_char_df
[없음] combined_history_df
[없음] rf_history_df
[없음] split_users_df

===== 검사 완료 =====


In [3]:

# 복구 1단계: 구글 드라이브 연결 및 저장 파일 확인

from google.colab import drive
from pathlib import Path

# 구글 드라이브 연결
drive.mount("/content/drive")

# 기존 캡스톤 데이터 경로
base_dir = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2"
)

folders = [
    base_dir / "genshin_processed",
    base_dir / "rf_model_data"
]

print("\n===== 저장된 데이터 확인 =====")

for folder in folders:
    print(f"\n📁 {folder.name}")

    if folder.exists():
        for file in sorted(folder.iterdir()):
            if file.is_file():
                print(" -", file.name)
    else:
        print("폴더 없음")

print("\n===== 확인 완료 =====")


Mounted at /content/drive

===== 저장된 데이터 확인 =====

📁 genshin_processed
 - genshin_analysis.parquet
 - genshin_analysis_clean.parquet
 - genshin_analysis_ready.parquet
 - genshin_train_normal_candidates.parquet
 - item_translation_full.csv
 - item_translation_map.csv
 - name_translation.csv
 - processing_summary.json
 - untranslated_names.csv

📁 rf_model_data
 - genshin_actual_rates.csv
 - genshin_official_rates.csv
 - genshin_probability_ci95.csv
 - genshin_probability_comparison.csv
 - genshin_probability_statistical_test.csv
 - genshin_user_bootstrap_ci.csv
 - rf_handoff_manifest.json
 - rf_normal_candidates.parquet
 - rf_split_users.csv
 - rf_tampered_item_raw.parquet

===== 확인 완료 =====


In [4]:

# 복구 2단계: 기존 학습 데이터 복원

import pandas as pd
from pathlib import Path

base_dir = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2"
)

rf_dir = base_dir / "rf_model_data"

# 1. 정상 후보 99,998건 불러오기
normal_df = pd.read_parquet(
    rf_dir / "rf_normal_candidates.parquet"
)

# 2. 기존 변조 1,000건 불러오기
tampered_df = pd.read_parquet(
    rf_dir / "rf_tampered_item_raw.parquet"
)

# 3. 사용자 분리 파일 불러오기
split_users_df = pd.read_csv(
    rf_dir / "rf_split_users.csv",
    dtype={"user_id": str}
)

# 4. 파일별 데이터 검증
print("===== 복구 2단계 결과 =====")
print("정상 후보:", len(normal_df))
print("기존 합성 변조:", len(tampered_df))
print("사용자 분리 정보:", len(split_users_df))

print("\n===== 정상 후보 컬럼 =====")
print(normal_df.columns.tolist())

print("\n===== 합성 변조 유형 =====")
if "tamper_type" in tampered_df.columns:
    print(tampered_df["tamper_type"].value_counts())

print("\n===== 복원 확인 =====")
print("정상 후보 99,998건:",
      len(normal_df) == 99998)
print("합성 변조 1,000건:",
      len(tampered_df) == 1000)
print("전체 사용자 16,289명:",
      len(split_users_df) == 16289)


===== 복구 2단계 결과 =====
정상 후보: 99998
기존 합성 변조: 1000
사용자 분리 정보: 16289

===== 정상 후보 컬럼 =====
['user_id', 'name_zh', 'name_ko', 'translation_ok', 'gacha_type', 'gacha_type_ko', 'item_type', 'item_type_ko', 'rank_type', 'gacha_id', 'gacha_time', 'is_five_star', 'is_four_star', 'gacha_date', 'gacha_month', 'gacha_id_num', 'pull_order', 'banner_pull_order', 'banner_group', 'previous_time', 'time_diff_seconds', 'group_order', 'pulls_since_last_5star', 'pity_before_pull', 'current_pity_count', 'pity_limit', 'pity_rule_violation', 'label']

===== 합성 변조 유형 =====
tamper_type
item_and_rank_change    1000
Name: count, dtype: int64

===== 복원 확인 =====
정상 후보 99,998건: True
합성 변조 1,000건: True
전체 사용자 16,289명: True


In [5]:

# 복구 3단계: 4성 캐릭터 변조 500건 재생성 및 저장

import pandas as pd
import numpy as np
from pathlib import Path

rng = np.random.default_rng(42)

# 1. 정상 후보에서 4성 캐릭터 500건 추출
target_char_df = normal_df.loc[
    (normal_df["rank_type"] == 4) &
    (normal_df["item_type_ko"] == "캐릭터")
].sample(n=500, random_state=42).copy()

# 2. 교체할 캐릭터 목록 생성
char_pool = (
    normal_df.loc[
        (normal_df["rank_type"] == 4) &
        (normal_df["item_type_ko"] == "캐릭터"),
        ["name_zh", "name_ko"]
    ]
    .drop_duplicates(subset=["name_ko"])
    .reset_index(drop=True)
)

new_names_zh = []
new_names_ko = []

# 3. 원래 캐릭터와 다른 4성 캐릭터 선택
for original_name in target_char_df["name_ko"]:
    candidates = char_pool.loc[
        char_pool["name_ko"] != original_name
    ]

    selected = candidates.iloc[
        rng.integers(len(candidates))
    ]

    new_names_zh.append(selected["name_zh"])
    new_names_ko.append(selected["name_ko"])

# 4. 별도 변조 복사본 생성
tampered_char_df = target_char_df.copy()

tampered_char_df["name_zh"] = new_names_zh
tampered_char_df["name_ko"] = new_names_ko
tampered_char_df["label"] = 1
tampered_char_df["tamper_type"] = "character_name_change"

# 5. 구글 드라이브에 저장
save_path = (
    Path("/content/drive/MyDrive/캡스톤 2/")
    / "capstonedata2/rf_model_data/"
    / "rf_tampered_character_raw.parquet"
)

tampered_char_df.to_parquet(
    save_path,
    index=False
)

# 6. 결과 검증
name_changed = (
    target_char_df["name_ko"].to_numpy()
    != tampered_char_df["name_ko"].to_numpy()
).sum()

print("===== 복구 3단계 결과 =====")
print("변조 기록:", len(tampered_char_df))
print("캐릭터 이름 변경:", name_changed)
print("4성 등급 유지:",
      (tampered_char_df["rank_type"] == 4).sum())

print("\n===== 파일 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("저장 경로:", save_path)

print("\n===== 기존 데이터 보존 =====")
print("정상 후보:", len(normal_df))
print("기존 3성→5성 변조:", len(tampered_df))


===== 복구 3단계 결과 =====
변조 기록: 500
캐릭터 이름 변경: 500
4성 등급 유지: 500

===== 파일 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/rf_tampered_character_raw.parquet

===== 기존 데이터 보존 =====
정상 후보: 99998
기존 3성→5성 변조: 1000


In [6]:

# 복구 4단계: 66~67단계 통합 이력 복원 및 저장

import pandas as pd
from pathlib import Path

KEY_COLS = ["user_id", "gacha_id"]

# 1. 변조 데이터 결합
rank_changes = tampered_df.copy()
rank_changes["tamper_type"] = "item_and_rank_change"

character_changes = tampered_char_df.copy()
character_changes["tamper_type"] = "character_name_change"

combined_tampered_df = pd.concat(
    [rank_changes, character_changes],
    ignore_index=True
)

assert len(combined_tampered_df) == 1500
assert not combined_tampered_df.duplicated(KEY_COLS).any()

# 2. 정상 원본 복사 및 변조 반영
rf_history_df = normal_df.copy().set_index(
    KEY_COLS, drop=False
)

replacement = combined_tampered_df.set_index(
    KEY_COLS, drop=False
)

assert replacement.index.isin(rf_history_df.index).all()

replace_cols = [
    "name_zh", "name_ko",
    "rank_type", "item_type", "item_type_ko",
    "is_five_star", "is_four_star"
]

for col in replace_cols:
    values = replacement[col].astype(rf_history_df[col].dtype)
    rf_history_df.loc[
        replacement.index, col
    ] = values.to_numpy()

# 3. 정상/변조 라벨 적용
rf_history_df["label"] = 0
rf_history_df["tamper_type"] = "normal"

rf_history_df.loc[replacement.index, "label"] = 1
rf_history_df.loc[
    replacement.index, "tamper_type"
] = replacement["tamper_type"].to_numpy()

rf_history_df = rf_history_df.reset_index(drop=True)

# 4. 사용자별/배너별 시간순 정렬
group_cols = ["user_id", "banner_group"]

rf_history_df = rf_history_df.sort_values(
    ["user_id", "banner_group", "gacha_time", "gacha_id_num"],
    kind="stable"
).reset_index(drop=True)

# 5. 천장 변수 재계산
rf_history_df["group_order"] = (
    rf_history_df.groupby(group_cols).cumcount()
)

five_positions = rf_history_df["group_order"].where(
    rf_history_df["rank_type"] == 5
)

last_five = five_positions.groupby(
    [
        rf_history_df["user_id"],
        rf_history_df["banner_group"]
    ]
).ffill()

rf_history_df["pulls_since_last_5star"] = (
    rf_history_df["group_order"] - last_five
)

rf_history_df["pity_before_pull"] = (
    rf_history_df.groupby(group_cols)[
        "pulls_since_last_5star"
    ].shift(1)
)

rf_history_df["current_pity_count"] = (
    rf_history_df["pity_before_pull"] + 1
)

pity_limits = {
    "character_event": 90,
    "weapon_event": 80,
    "standard": 90
}

rf_history_df["pity_limit"] = (
    rf_history_df["banner_group"].map(pity_limits)
)

rf_history_df["pity_rule_violation"] = (
    rf_history_df["current_pity_count"]
    > rf_history_df["pity_limit"]
).fillna(False)

# 6. 결과 검증
assert len(rf_history_df) == 99998
assert (rf_history_df["label"] == 1).sum() == 1500
assert not rf_history_df.duplicated(KEY_COLS).any()
assert (
    rf_history_df["is_five_star"]
    == (rf_history_df["rank_type"] == 5)
).all()

# 7. 완성 데이터 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "rf_train_history_recovered.parquet"
)

rf_history_df.to_parquet(save_path, index=False)

print("===== 복구 4단계 결과 =====")
print("전체 기록:", len(rf_history_df))
print("정상 후보:", (rf_history_df["label"] == 0).sum())
print("합성 변조:", (rf_history_df["label"] == 1).sum())

print("\n===== 변조 유형 =====")
print(rf_history_df["tamper_type"].value_counts())

print("\n===== 천장 검사 =====")
print("5성 기록:", (rf_history_df["rank_type"] == 5).sum())
print(
    "천장 초과 의심:",
    int(rf_history_df["pity_rule_violation"].sum())
)
print(
    "직전 천장 상태 미확인:",
    int(rf_history_df["pity_before_pull"].isna().sum())
)

print("\n===== 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("저장 경로:", save_path)


===== 복구 4단계 결과 =====
전체 기록: 99998
정상 후보: 98498
합성 변조: 1500

===== 변조 유형 =====
tamper_type
normal                   98498
item_and_rank_change      1000
character_name_change      500
Name: count, dtype: int64

===== 천장 검사 =====
5성 기록: 2649
천장 초과 의심: 0
직전 천장 상태 미확인: 12287

===== 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/rf_train_history_recovered.parquet


In [7]:

# 74단계: 랜덤 포레스트 입력 변수 점검

import pandas as pd

basic_features = [
    "gacha_type",
    "item_type",
    "rank_type",
    "name_ko",
    "banner_group",
    "time_diff_seconds",
    "pity_before_pull"
]

forbidden_features = [
    "user_id",
    "gacha_id",
    "label",
    "tamper_type",
    "pulls_since_last_5star",
    "pity_rule_violation"
]

rows = []

for col in basic_features:
    assert col in rf_history_df.columns, (
        f"누락된 컬럼: {col}"
    )

    rows.append({
        "변수": col,
        "자료형": str(rf_history_df[col].dtype),
        "결측값": int(rf_history_df[col].isna().sum()),
        "고유값 수": int(rf_history_df[col].nunique())
    })

feature_overview_df = pd.DataFrame(rows)

print("===== 74단계: 입력 변수 점검 =====")
display(feature_overview_df)

print("\n===== 변조 유형별 아이템 종류 =====")
display(
    pd.crosstab(
        rf_history_df["tamper_type"],
        rf_history_df["item_type_ko"]
    )
)

print("\n===== 입력 안전성 검사 =====")
print("입력 후보:", len(basic_features))
print(
    "금지 변수 혼입:",
    len(set(basic_features) & set(forbidden_features))
)

print("\n===== 데이터 확인 =====")
print("전체 기록:", len(rf_history_df))
print("합성 변조:", int(rf_history_df["label"].sum()))


===== 74단계: 입력 변수 점검 =====


,변수,자료형,결측값,고유값 수
0,gacha_type,object,0,5
1,item_type,int8,0,2
2,rank_type,int8,0,3
3,name_ko,object,0,117
4,banner_group,object,0,4
5,time_diff_seconds,float64,383,17183
6,pity_before_pull,float64,12287,84



===== 변조 유형별 아이템 종류 =====


item_type_ko,무기,캐릭터
tamper_type,,
character_name_change,0,500
item_and_rank_change,1000,0
normal,89999,8499



===== 입력 안전성 검사 =====
입력 후보: 7
금지 변수 혼입: 0

===== 데이터 확인 =====
전체 기록: 99998
합성 변조: 1500


In [8]:

# 75단계: 랜덤 포레스트 입력 데이터 준비

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. 모델 입력 변수
categorical_features = [
    "gacha_type",
    "name_ko",
    "banner_group"
]

numeric_features = [
    "item_type",
    "rank_type",
    "time_diff_seconds"
]

model_features = (
    categorical_features + numeric_features
)

# 2. 입력 데이터와 정답 분리
X_train = rf_history_df[model_features].copy()
y_train = rf_history_df["label"].astype(int).copy()

# 3. 숫자형 결측값 처리
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# 4. 문자형 변수 숫자 변환
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# 5. 전처리 도구 구성
preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_transformer,
        categorical_features
    ),
    (
        "numeric",
        numeric_transformer,
        numeric_features
    )
])

# 6. 결과 검사
print("===== 75단계: 모델 입력 데이터 =====")
print("전체 기록:", len(X_train))
print("입력 변수 수:", len(model_features))

print("\n===== 정답 분포 =====")
print(y_train.value_counts().sort_index())

print("\n===== 입력 변수 =====")
print(model_features)

print("\n===== 결측값 =====")
print(X_train.isna().sum())

print("\n===== 전처리 도구 준비 =====")
print("전처리 객체:", type(preprocessor).__name__)
print("학습 실행 여부: 아직 실행하지 않음")


===== 75단계: 모델 입력 데이터 =====
전체 기록: 99998
입력 변수 수: 6

===== 정답 분포 =====
label
0    98498
1     1500
Name: count, dtype: int64

===== 입력 변수 =====
['gacha_type', 'name_ko', 'banner_group', 'item_type', 'rank_type', 'time_diff_seconds']

===== 결측값 =====
gacha_type             0
name_ko                0
banner_group           0
item_type              0
rank_type              0
time_diff_seconds    383
dtype: int64

===== 전처리 도구 준비 =====
전처리 객체: ColumnTransformer
학습 실행 여부: 아직 실행하지 않음


In [9]:

# 76단계: 랜덤 포레스트 모델 학습 및 저장

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from pathlib import Path
import joblib
import time

# 1. 랜덤 포레스트 모델 설정
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# 2. 전처리 + 랜덤 포레스트 연결
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf_model)
])

# 3. 모델 학습
print("===== 76단계: 랜덤 포레스트 학습 =====")

start_time = time.time()

rf_pipeline.fit(X_train, y_train)

elapsed = time.time() - start_time

print("학습 완료")
print(f"학습 소요 시간: {elapsed:.1f}초")

# 4. 모델 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "genshin_random_forest_v1.joblib"
)

joblib.dump(rf_pipeline, save_path)

print("\n===== 모델 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("저장 경로:", save_path)

# 5. 학습 결과 기본 확인
print("\n===== 모델 정보 =====")
print("모델:", type(rf_model).__name__)
print("트리 개수:", rf_model.n_estimators)
print("학습 데이터:", len(X_train))
print("입력 변수:", len(model_features))
print("학습 사용자:", rf_history_df["user_id"].nunique())


===== 76단계: 랜덤 포레스트 학습 =====
학습 완료
학습 소요 시간: 18.7초

===== 모델 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/genshin_random_forest_v1.joblib

===== 모델 정보 =====
모델: RandomForestClassifier
트리 개수: 300
학습 데이터: 99998
입력 변수: 6
학습 사용자: 116


In [10]:

# 77단계: 검증·테스트 사용자 원본 데이터 추출 및 저장

import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

base_dir = Path(
    "/content/drive/MyDrive/캡스톤 2/capstonedata2"
)

source_path = (
    base_dir / "genshin_processed"
    / "genshin_analysis_ready.parquet"
)

output_dir = base_dir / "rf_model_data"
output_dir.mkdir(parents=True, exist_ok=True)

# 사용자 구분
val_users = set(
    split_users_df.loc[
        split_users_df["split"] == "val", "user_id"
    ]
)

test_users = set(
    split_users_df.loc[
        split_users_df["split"] == "test", "user_id"
    ]
)

train_users = set(
    split_users_df.loc[
        split_users_df["split"] == "train", "user_id"
    ]
)

assert val_users.isdisjoint(test_users)
assert val_users.isdisjoint(train_users)
assert test_users.isdisjoint(train_users)

# 원본 파일을 묶음 단위로 읽기
pf = pq.ParquetFile(source_path)

val_parts = []
test_parts = []

for i in range(pf.num_row_groups):
    batch = pf.read_row_group(i).to_pandas()
    batch["user_id"] = batch["user_id"].astype(str)

    val_part = batch.loc[
        batch["user_id"].isin(val_users)
    ]
    test_part = batch.loc[
        batch["user_id"].isin(test_users)
    ]

    if len(val_part):
        val_parts.append(val_part.copy())

    if len(test_part):
        test_parts.append(test_part.copy())

    if (i + 1) % 10 == 0:
        print(f"읽기 완료: {i+1}/{pf.num_row_groups}")

# 각 그룹 결합
val_raw_df = pd.concat(
    val_parts, ignore_index=True
)
test_raw_df = pd.concat(
    test_parts, ignore_index=True
)

# 사용자 소속 검사
assert set(val_raw_df["user_id"]).issubset(val_users)
assert set(test_raw_df["user_id"]).issubset(test_users)

assert set(val_raw_df["user_id"]).isdisjoint(
    set(test_raw_df["user_id"])
)

# 드라이브에 저장
val_path = output_dir / "rf_validation_raw.parquet"
test_path = output_dir / "rf_test_raw.parquet"

val_raw_df.to_parquet(val_path, index=False)
test_raw_df.to_parquet(test_path, index=False)

print("\n===== 77단계: 평가 데이터 준비 =====")
print("검증 기록:", f"{len(val_raw_df):,}")
print("검증 사용자:", val_raw_df["user_id"].nunique())
print("테스트 기록:", f"{len(test_raw_df):,}")
print("테스트 사용자:", test_raw_df["user_id"].nunique())

print("\n===== 저장 확인 =====")
print("검증 데이터 저장:", val_path.exists())
print("테스트 데이터 저장:", test_path.exists())

print("\n===== 원본 유지 =====")
print("학습 데이터:", len(rf_history_df))
print("학습 모델 저장:", (
    output_dir / "genshin_random_forest_v1.joblib"
).exists())


읽기 완료: 10/60
읽기 완료: 20/60
읽기 완료: 30/60
읽기 완료: 40/60
읽기 완료: 50/60
읽기 완료: 60/60

===== 77단계: 평가 데이터 준비 =====
검증 기록: 2,411,948
검증 사용자: 2443
테스트 기록: 2,436,782
테스트 사용자: 2444

===== 저장 확인 =====
검증 데이터 저장: True
테스트 데이터 저장: True

===== 원본 유지 =====
학습 데이터: 99998
학습 모델 저장: True


In [11]:

# 78단계: 검증용 원본 데이터 구조 및 품질 확인

import pandas as pd

print("===== 78단계: 검증용 원본 검사 =====")

print("전체 기록:", f"{len(val_raw_df):,}")
print("사용자 수:", val_raw_df["user_id"].nunique())

print("\n===== 컬럼 및 자료형 =====")
print(val_raw_df.dtypes.to_string())

print("\n===== 주요 항목 결측값 =====")
check_cols = [
    "user_id",
    "gacha_id",
    "gacha_time",
    "gacha_type",
    "rank_type",
    "item_type",
    "name_ko"
]

print(val_raw_df[check_cols].isna().sum())

print("\n===== 등급별 기록 =====")
print(val_raw_df["rank_type"].value_counts().sort_index())

print("\n===== 기원 종류 =====")
print(val_raw_df["gacha_type"].astype(str).value_counts())

print("\n===== 4성 캐릭터 기록 =====")
four_char_count = (
    (val_raw_df["rank_type"] == 4)
    & (val_raw_df["item_type_ko"] == "캐릭터")
).sum()

print("4성 캐릭터:", int(four_char_count))

print("\n===== 중복 식별자 =====")
print(
    val_raw_df.duplicated(
        ["user_id", "gacha_id"]
    ).sum()
)

print("\n===== 테스트 원본 보존 =====")
print("테스트 기록:", f"{len(test_raw_df):,}")
print("테스트 사용자:", test_raw_df["user_id"].nunique())


===== 78단계: 검증용 원본 검사 =====
전체 기록: 2,411,948
사용자 수: 2443

===== 컬럼 및 자료형 =====
user_id                   object
name_zh                   object
name_ko                   object
translation_ok              bool
gacha_type                object
gacha_type_ko             object
item_type                   int8
item_type_ko              object
rank_type                   int8
gacha_id                  object
gacha_time        datetime64[us]
is_five_star                bool
is_four_star                bool
gacha_date                object
gacha_month               object

===== 주요 항목 결측값 =====
user_id       0
gacha_id      0
gacha_time    0
gacha_type    0
rank_type     0
item_type     0
name_ko       0
dtype: int64

===== 등급별 기록 =====
rank_type
3    2049758
4     322417
5      39773
Name: count, dtype: int64

===== 기원 종류 =====
gacha_type
301    1265263
302     457244
200     351381
400     321420
100      16640
Name: count, dtype: int64

===== 4성 캐릭터 기록 =====
4성 캐릭터: 185927

===== 중복 식별자 

In [12]:

# 79단계: 검증용 합성 변조 1,500건 생성 및 저장

import pandas as pd
import numpy as np
from pathlib import Path

rng = np.random.default_rng(79)

KEY_COLS = ["user_id", "gacha_id"]

# 1. 검증 원본에서 변조 대상 선정
rank_targets = val_raw_df.loc[
    (val_raw_df["rank_type"] == 3)
    & (val_raw_df["item_type_ko"] == "무기")
].sample(n=1000, random_state=79).copy()

char_targets = val_raw_df.loc[
    (val_raw_df["rank_type"] == 4)
    & (val_raw_df["item_type_ko"] == "캐릭터")
].sample(n=500, random_state=80).copy()

# 2. 5성 무기 후보: 학습 원본에서 추출
weapon_pool = (
    normal_df.loc[
        (normal_df["rank_type"] == 5)
        & (normal_df["item_type_ko"] == "무기"),
        ["name_zh", "name_ko"]
    ]
    .drop_duplicates(subset=["name_ko"])
    .reset_index(drop=True)
)

# 3. 4성 캐릭터 후보: 학습 원본에서 추출
char_pool = (
    normal_df.loc[
        (normal_df["rank_type"] == 4)
        & (normal_df["item_type_ko"] == "캐릭터"),
        ["name_zh", "name_ko"]
    ]
    .drop_duplicates(subset=["name_ko"])
    .reset_index(drop=True)
)

assert len(weapon_pool) > 0
assert len(char_pool) > 1

# 4. 3성 무기 → 5성 무기 변조
rank_tampered_val = rank_targets.copy()

weapon_choices = rng.integers(
    0, len(weapon_pool), size=len(rank_targets)
)

rank_tampered_val["name_zh"] = (
    weapon_pool.iloc[weapon_choices]["name_zh"].to_numpy()
)
rank_tampered_val["name_ko"] = (
    weapon_pool.iloc[weapon_choices]["name_ko"].to_numpy()
)

rank_tampered_val["rank_type"] = 5
rank_tampered_val["is_five_star"] = True
rank_tampered_val["is_four_star"] = False
rank_tampered_val["label"] = 1
rank_tampered_val["tamper_type"] = "item_and_rank_change"

# 5. 4성 캐릭터 → 다른 4성 캐릭터 변조
char_tampered_val = char_targets.copy()

new_zh = []
new_ko = []

for old_name in char_targets["name_ko"]:
    candidates = char_pool.loc[
        char_pool["name_ko"] != old_name
    ]
    selected = candidates.iloc[
        rng.integers(len(candidates))
    ]
    new_zh.append(selected["name_zh"])
    new_ko.append(selected["name_ko"])

char_tampered_val["name_zh"] = new_zh
char_tampered_val["name_ko"] = new_ko
char_tampered_val["label"] = 1
char_tampered_val["tamper_type"] = "character_name_change"

# 6. 합성 변조 데이터 합치기
val_tampered_df = pd.concat(
    [rank_tampered_val, char_tampered_val],
    ignore_index=True
)

# 7. 중복 및 데이터 검증
assert len(val_tampered_df) == 1500
assert not val_tampered_df.duplicated(KEY_COLS).any()

assert (
    rank_targets["name_ko"].to_numpy()
    != rank_tampered_val["name_ko"].to_numpy()
).all()

assert (
    char_targets["name_ko"].to_numpy()
    != char_tampered_val["name_ko"].to_numpy()
).all()

# 8. 구글 드라이브 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "rf_validation_tampered.parquet"
)

val_tampered_df.to_parquet(save_path, index=False)

print("===== 79단계: 검증용 합성 변조 =====")
print("전체 변조:", len(val_tampered_df))
print("\n변조 유형별 기록:")
print(val_tampered_df["tamper_type"].value_counts())

print("\n===== 검증 =====")
print("중복 기록:", val_tampered_df.duplicated(KEY_COLS).sum())
print("검증 사용자 수:", val_tampered_df["user_id"].nunique())

print("\n===== 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("저장 경로:", save_path)

print("\n===== 테스트 데이터 보존 =====")
print("테스트 원본:", len(test_raw_df))



===== 79단계: 검증용 합성 변조 =====
전체 변조: 1500

변조 유형별 기록:
tamper_type
item_and_rank_change     1000
character_name_change     500
Name: count, dtype: int64

===== 검증 =====
중복 기록: 0
검증 사용자 수: 961

===== 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/rf_validation_tampered.parquet

===== 테스트 데이터 보존 =====
테스트 원본: 2436782


In [13]:

# 80단계: 검증용 합성 변조 이력 생성 및 저장

import pandas as pd
from pathlib import Path

KEY_COLS = ["user_id", "gacha_id"]

# 1. 검증용 원본 복사
val_history_df = val_raw_df.copy()

# 2. 변조 대상 중복 및 연결 확인
assert not val_history_df.duplicated(KEY_COLS).any()
assert not val_tampered_df.duplicated(KEY_COLS).any()

val_history_df = val_history_df.set_index(
    KEY_COLS, drop=False
)

replacement = val_tampered_df.set_index(
    KEY_COLS, drop=False
)

assert replacement.index.isin(
    val_history_df.index
).all(), "원본에 없는 변조 기록이 있습니다."

# 3. 변경된 컬럼만 반영
replace_cols = [
    "name_zh", "name_ko",
    "rank_type",
    "is_five_star", "is_four_star"
]

for col in replace_cols:
    values = replacement[col].astype(
        val_history_df[col].dtype
    )

    val_history_df.loc[
        replacement.index, col
    ] = values.to_numpy()

# 4. 정상/변조 라벨 적용
val_history_df["label"] = 0
val_history_df["tamper_type"] = "normal"

val_history_df.loc[
    replacement.index, "label"
] = 1

val_history_df.loc[
    replacement.index, "tamper_type"
] = replacement["tamper_type"].to_numpy()

val_history_df = val_history_df.reset_index(drop=True)

# 5. 배너 그룹 생성
banner_map = {
    "301": "character_event",
    "400": "character_event",
    "302": "weapon_event",
    "200": "standard",
    "100": "beginner"
}

val_history_df["banner_group"] = (
    val_history_df["gacha_type"]
    .astype(str).map(banner_map)
)

# 6. 사용자별 뽑기 시간 차이 계산
val_history_df = val_history_df.sort_values(
    ["user_id", "gacha_time", "gacha_id"],
    kind="stable"
).reset_index(drop=True)

val_history_df["time_diff_seconds"] = (
    val_history_df.groupby("user_id")[
        "gacha_time"
    ].diff().dt.total_seconds()
)

# 7. 검증
print("===== 80단계: 검증용 이력 =====")
print("전체 기록:", f"{len(val_history_df):,}")
print("정상 후보:", f"{(val_history_df['label'] == 0).sum():,}")
print("합성 변조:", f"{(val_history_df['label'] == 1).sum():,}")

print("\n===== 변조 유형 =====")
print(val_history_df["tamper_type"].value_counts())

print("\n===== 입력 변수 검사 =====")
print("배너 그룹 결측값:",
      val_history_df["banner_group"].isna().sum())
print("시간 차이 결측값:",
      val_history_df["time_diff_seconds"].isna().sum())

print("\n===== 원본 일관성 검사 =====")
print("중복 식별자:",
      val_history_df.duplicated(KEY_COLS).sum())
print("5성 표시 불일치:",
      (val_history_df["is_five_star"]
       != (val_history_df["rank_type"] == 5)).sum())
print("4성 표시 불일치:",
      (val_history_df["is_four_star"]
       != (val_history_df["rank_type"] == 4)).sum())

# 8. 저장
save_path = Path(
    "/content/drive/MyDrive/캡스톤 2/"
    "capstonedata2/rf_model_data/"
    "rf_validation_history.parquet"
)

val_history_df.to_parquet(save_path, index=False)

print("\n===== 저장 확인 =====")
print("저장 성공:", save_path.exists())
print("저장 경로:", save_path)


===== 80단계: 검증용 이력 =====
전체 기록: 2,411,948
정상 후보: 2,410,448
합성 변조: 1,500

===== 변조 유형 =====
tamper_type
normal                   2410448
item_and_rank_change        1000
character_name_change        500
Name: count, dtype: int64

===== 입력 변수 검사 =====
배너 그룹 결측값: 0
시간 차이 결측값: 2443

===== 원본 일관성 검사 =====
중복 식별자: 0
5성 표시 불일치: 0
4성 표시 불일치: 0

===== 저장 확인 =====
저장 성공: True
저장 경로: /content/drive/MyDrive/캡스톤 2/capstonedata2/rf_model_data/rf_validation_history.parquet


In [14]:

# 81단계: 학습·검증 데이터의 시간 차이 계산 기준 확인

import pandas as pd
import numpy as np

# 1. 학습 데이터에서 시간 차이를 다시 계산
train_check = rf_history_df.copy()

# 사용자 기준
train_check = train_check.sort_values(
    ["user_id", "gacha_time", "gacha_id"],
    kind="stable"
)

train_check["diff_by_user"] = (
    train_check.groupby("user_id")["gacha_time"]
    .diff().dt.total_seconds()
)

# 사용자 + 배너 그룹 기준
train_check["diff_by_banner"] = (
    train_check.groupby(
        ["user_id", "banner_group"]
    )["gacha_time"]
    .diff().dt.total_seconds()
)

# 결측값끼리는 동일한 것으로 취급
def equal_with_nan(a, b):
    return (
        np.isclose(
            a.to_numpy(dtype=float),
            b.to_numpy(dtype=float),
            equal_nan=True
        )
    ).sum()

user_match = equal_with_nan(
    train_check["time_diff_seconds"],
    train_check["diff_by_user"]
)

banner_match = equal_with_nan(
    train_check["time_diff_seconds"],
    train_check["diff_by_banner"]
)

print("===== 81단계: 시간 차이 계산 기준 검사 =====")

print("학습 기록:", len(train_check))
print("학습 사용자:", train_check["user_id"].nunique())

print("\n===== 결측값 비교 =====")
print(
    "기존 학습 시간 차이:",
    train_check["time_diff_seconds"].isna().sum()
)
print(
    "사용자 기준 재계산:",
    train_check["diff_by_user"].isna().sum()
)
print(
    "사용자+배너 기준 재계산:",
    train_check["diff_by_banner"].isna().sum()
)

print("\n===== 기존 값과 일치하는 기록 =====")
print("사용자 기준:", user_match)
print("사용자+배너 기준:", banner_match)

print("\n===== 검증 데이터 =====")
print("검증 기록:", len(val_history_df))
print(
    "검증 시간 차이 결측:",
    val_history_df["time_diff_seconds"].isna().sum()
)

print("\n===== 참고 =====")
print("이번 단계에서는 데이터를 변경하지 않았습니다.")


===== 81단계: 시간 차이 계산 기준 검사 =====
학습 기록: 99998
학습 사용자: 116

===== 결측값 비교 =====
기존 학습 시간 차이: 383
사용자 기준 재계산: 116
사용자+배너 기준 재계산: 383

===== 기존 값과 일치하는 기록 =====
사용자 기준: 91471
사용자+배너 기준: 99998

===== 검증 데이터 =====
검증 기록: 2411948
검증 시간 차이 결측: 2443

===== 참고 =====
이번 단계에서는 데이터를 변경하지 않았습니다.
